In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S10_clasificadores_clasicos"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 10 — Clasificadores clásicos: LDA/QDA + Naive Bayes + clases desbalanceadas

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios

> Curso **Herramientas para la Ciencia de Datos**, Facultad de Negocios, Administración y Ciencia de Datos para Negocios — UPC.
> Español impersonal, fechas DD/MM/YYYY, todas las cifras provienen de la investigación verificada de la sesión (`el material de referencia de la sesión`).

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


## 1. Objetivos de aprendizaje
Al terminar la sesión, el estudiante:

- Distingue el enfoque **generativo** (LDA, QDA, Naive Bayes) del **discriminativo** (regresión logística de S09), ajusta estos modelos y comprende sus **supuestos** y **fronteras de decisión**.
- Diagnostica y trata el **desbalance de clases** con **SMOTE**, ponderación (`class_weight`) y submuestreo.
- Elige la **métrica adecuada** (precision, recall, F1, **PR-AUC**, log-loss) cuando la exactitud (accuracy) induce a error.
- Construye un **benchmark reproducible** de clasificadores y selecciona el **modelo campeón** con su umbral.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —10.1 a 10.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 9)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **10.1** | ¿Por qué la regresión logística no siempre es el mejor clasificador? | Sin celdas: se abre en clase con el límite de la logística de S09 |
| **10.2** | ¿Qué distingue un modelo generativo de uno discriminativo? | «Teoría guiada» — Bayes, LDA, QDA, Naive Bayes, desbalance y PR-AUC |
| **10.3** | ¿Cómo clasifica cada uno de estos métodos? | «Teoría guiada» — Bayes, LDA, QDA, Naive Bayes, desbalance y PR-AUC |
| **10.4** | ¿Cómo se mide cuando una clase representa el 1 % de los casos? | «Teoría guiada» — Bayes, LDA, QDA, Naive Bayes, desbalance y PR-AUC |
| **10.5** | ¿Se sostiene con datos reales? La réplica de Fisher (1936) | «La réplica de Fisher (1936) sobre Iris» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **10.6** | ¿Cuándo se puede confiar en estos clasificadores? | «Supuestos: decisiones y condiciones de validez» |
| **10.7** | ¿Cómo se verifica que el resultado es real? | «Verificación desde la base» |
| **10.8** | ¿Qué decisión habilita? Detección de fraude con proporción 99:1 | «Detección de fraude con tarjeta (ULB, 99:1)» — laboratorio, paso 6 |
| **10.9** | ¿Qué no se puede afirmar, y qué sigue en S11? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **corre**: **explica y descompone** cada paso. Los marcadores guían la lectura:

- **❓ Qué se quiere averiguar** — abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** — antes de cada celda: qué va a calcular y por qué.
- **📖 Cómo se lee esta salida** — después de una salida numérica clave: cómo interpretarla en negocio.
- **💡 Intuición** y **⚠️ Alerta / supuesto** — matices y errores a evitar.
- **🖐️ Cálculo manual** — se reconstruye la mecánica (regla discriminante de LDA; precisión/recall) y se verifica contra la librería con `assert`.
- **✅ Verificación desde la base** — se **recomputa** el resultado clave desde los datos y se cruza con el Excel (`assert`).
- **🧮 Matemática en el cuerpo** — la fórmula (Bayes/Naive Bayes, discriminante de Fisher, LDA vs QDA, precisión/recall/F1/PR-AUC) donde se aplica.
- **🧱 Construcción desde cero** — se rearma el flujo del desbalance (split → SMOTE solo en train → métricas en test) y se reproduce el contrato (`assert`).
- **📄 En el paper** — procedencia exacta del resultado replicado (autor, año, publicación).

**Operativo vs. benchmark (regla de oro de la sesión).** El **valor OPERATIVO** es el que produce este cuaderno con el venv (scikit-learn 1.6.1 / imbalanced-learn 0.14.2) y queda en `resultados/S10_resultados.xlsx`. Cualquier cifra publicada (RepeatedStratifiedKFold de la literatura, PR-AUC o recall post-SMOTE de otros pipelines) se cita **etiquetada como benchmark publicado (fuente, otro pipeline)**, nunca como resultado propio.

**Convención Excel.** Los resultados y pruebas se vuelcan a `resultados/S10_resultados.xlsx` y las **figuras de resultados se generan LEYENDO ese Excel**. La **verificación desde la base (6.1)** y los **diagnósticos de supuestos (Sección 8) NO escriben en el Excel**.

**Mapa de celdas ↔ slides:** el cuaderno de la sesión. **Supuestos (fuente canónica):** la guía de supuestos de la sesión.

## Preparación del entorno

La primera celda instala las librerías **solo en Google Colab**, con las versiones **fijadas** de la sesión. En ejecución local se salta automáticamente.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q imbalanced-learn

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "scikit-learn": "1.6.1",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Carga las librerías de la sesión (LDA/QDA, Naive Bayes, métricas, SMOTE), fija la **semilla 42** (reproducibilidad), define la **paleta UPC** y el ayudante `mostrar()` que guarda cada figura a 150 dpi y la muestra. Los avisos de covarianza/convergencia son **esperados**, no errores.

In [ ]:
# Configuración e imports
import warnings
warnings.filterwarnings("ignore")   # avisos de covarianza/convergencia: esperados, no son errores

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             precision_score, recall_score, f1_score, log_loss)
from imblearn.over_sampling import SMOTE

import matplotlib
matplotlib.use("Agg")               # backend headless: cada figura se guarda como PNG y se muestra
import matplotlib.pyplot as plt
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta del curso (UPC)
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
PALETA = [UPC_ROJO, UPC_TINTA, "#E4879C", "#5B6472", "#A31621", "#B0B3B5", "#7A8CA3"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG a 150 dpi y la muestra en el cuaderno (backend Agg)."
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

import sklearn, imblearn
print("Librerías cargadas. scikit-learn", sklearn.__version__,
      "| imbalanced-learn", imblearn.__version__, "| listo para LDA/QDA, Naive Bayes y desbalance.")

**🔎 Qué hace este código.** Localiza la carpeta de la sesión (funciona en local, en nbconvert y en Colab), fija las rutas de `data/`, `resultados/` y `figuras/`, e importa los **cargadores oficiales** `obtener_iris()` (Fisher 1936, offline) y `cargar_fraude_completo()` (ULB 284 807×31, se cachea fuera de OneDrive en runtime). El desbalance 99:1 real solo vive en el completo.

In [ ]:
# Localización de carpetas de la sesión (funciona en local, en nbconvert y en Colab)
def localizar_sesion():
    aqui = Path.cwd()
    for base in [aqui, *aqui.parents]:
        if base.name == "S10_clasificadores_clasicos" and base.name.startswith("S10"):
            return base
        cand = base / "Sesiones" / "S10_clasificadores_clasicos"
        if cand.exists():
            return cand
        if base.name.startswith("S10"):
            return base
    return None

SESION = localizar_sesion()
if SESION is None:
    SESION = Path("/content/S10_clasificadores_clasicos")
    (SESION / "data").mkdir(parents=True, exist_ok=True)
    print("Modo Colab: carpeta de trabajo en", SESION)
else:
    print("Sesión localizada en:", SESION)

DATA = SESION / "data"
RESULTADOS = SESION / "resultados"; RESULTADOS.mkdir(exist_ok=True)
FIGURAS = SESION / "figuras"; FIGURAS.mkdir(exist_ok=True)
XLSX = RESULTADOS / "S10_resultados.xlsx"

# --- Cargadores OFICIALES de la sesión (data/descargar_datos.py) ---
# obtener_iris()          -> Iris 150x5 (Fisher 1936), offline (sklearn built-in).
# cargar_fraude_completo()-> fraude ULB COMPLETO 284807x31 (492 fraudes, 0,17 %).
#   El completo pesa ~150 MB: NUNCA se guarda en OneDrive; se cachea fuera, en runtime,
#   desde un mirror abierto. La muestra local `creditcard_muestra.csv` (1842 filas,
#   ~26,7 % de fraude) es SOLO para inspección offline; el desbalance real 99:1 se
#   trabaja SIEMPRE sobre el COMPLETO (targets #3-#5 de la réplica).
if str(DATA) not in sys.path:
    sys.path.insert(0, str(DATA))
try:
    from descargar_datos import obtener_iris, cargar_fraude_completo
    print("Cargadores importados de data/descargar_datos.py")
except Exception as e:
    # Fallback (Colab sin el .py en el path): cargadores mínimos equivalentes.
    print("descargar_datos.py no disponible; usando cargadores de respaldo:", e)
    import tempfile, os
    _CC_URL = ("https://huggingface.co/datasets/David-Egea/"
               "Creditcard-fraud-detection/resolve/main/creditcard.csv")
    def obtener_iris(verify_uci=False):
        d = load_iris(as_frame=True)
        df = d.data.copy()
        df.columns = ["sepal_length_cm", "sepal_width_cm",
                      "petal_length_cm", "petal_width_cm"]
        df["species"] = pd.Categorical.from_codes(
            d.target, ["setosa", "versicolor", "virginica"])
        return df
    def cargar_fraude_completo():
        cache = os.path.join(tempfile.gettempdir(), "curso_upc_s10_creditcard")
        os.makedirs(cache, exist_ok=True)
        p = os.path.join(cache, "creditcard.csv")
        if not os.path.exists(p):
            print("Descargando el dataset COMPLETO de fraude (~150 MB, solo runtime)...")
            pd.read_csv(_CC_URL).to_csv(p, index=False)
        return pd.read_csv(p)

FEATS = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]
ESPECIES = ["setosa", "versicolor", "virginica"]
print("Resultados ->", RESULTADOS)
print("Figuras    ->", FIGURAS)

## 10.2 a 10.4 — ¿Qué distingue un modelo generativo de uno discriminativo, cómo clasifica cada método y cómo se mide con clases desbalanceadas? Teoría guiada (Sección 3 del cuaderno)

Bloques cortos con una **lectura de negocio** al final de cada uno. El detalle está en el glosario de la sesión e la guía de interpretación de resultados.

### Generativo vs. discriminativo y el teorema de Bayes — capítulo 10.2 (subsección 3.1)

Un clasificador **generativo** modela **cómo se generan** los datos de cada clase: la verosimilitud `P(x | clase)` y el prior `P(clase)`. Con el **teorema de Bayes** obtiene la posterior y asigna la clase más probable:

$$P(\text{clase}\mid x)\;\propto\;P(x\mid \text{clase})\,P(\text{clase}) \qquad \hat{y}=\arg\max_{\text{clase}} P(\text{clase}\mid x)$$

LDA, QDA y Naive Bayes son **generativos**. La regresión logística (S09) es **discriminativa**: modela `P(clase | x)` directamente, sin describir la distribución de las variables.

#### 🧮 Matemática en el cuerpo — la regla de Bayes (generativo vs. discriminativo) y Naive Bayes

Un clasificador **generativo** modela **cómo se generan** los datos de cada clase y aplica el **teorema de Bayes**:

$$P(y\mid x) \;=\; \frac{P(x\mid y)\,P(y)}{P(x)} \;\propto\; P(x\mid y)\,P(y), \qquad \hat{y} \;=\; \arg\max_{y}\; P(x\mid y)\,P(y)$$

- $P(x\mid y)$ es la **verosimilitud** (cómo se distribuyen las variables dentro de la clase) y $P(y)$ el **prior** (proporción de la clase). LDA, QDA y Naive Bayes son **generativos**: estiman $P(x\mid y)$ y $P(y)$.
- La **regresión logística (S09) es discriminativa**: modela $P(y\mid x)$ **directamente**, sin describir la distribución de $x$.
- **Naive Bayes** factoriza la verosimilitud asumiendo **independencia condicional** de las variables dada la clase:

$$P(x\mid y) \;=\; \prod_{j=1}^{p} P(x_j\mid y)$$

Fuente: el glosario de la sesión «Sección 1», «Sección 2» y «Sección 5»; la guía de supuestos de la sesión Parte 1 y Parte 4.1.

**🔎 Qué hace este código.** Ajusta una **gaussiana por especie** a la variable *largo del pétalo* y traza sus densidades $P(x\mid \text{clase})$: es la **visión generativa**. Bayes clasifica cada flor con la especie cuya densidad (por su prior) es mayor en ese punto.

In [ ]:
# Mini-demo: la visión generativa. Ajustar una gaussiana por especie a 'petal length'
iris_demo = obtener_iris()
x_pl = iris_demo["petal_length_cm"].values
fig, ax = plt.subplots(figsize=(6.4, 3.4))
rejilla = np.linspace(0.5, 7.5, 400)
for i, esp in enumerate(ESPECIES):
    xi = x_pl[iris_demo["species"].astype(str).values == esp]
    mu, sd = xi.mean(), xi.std(ddof=1)
    dens = np.exp(-0.5 * ((rejilla - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    ax.plot(rejilla, dens, color=PALETA[i], lw=2.2, label=f"P(x | {esp})")
    ax.axvline(mu, color=PALETA[i], ls=":", lw=1)
ax.set_xlabel("Largo del pétalo (cm)")
ax.set_ylabel("Densidad  P(x | clase)")
ax.set_title("Visión generativa: una gaussiana por especie (Bayes elige la de mayor posterior)")
ax.legend(frameon=False)
mostrar(fig, FIGURAS / "demo_generativo_gaussianas.png")
print("La regla de Bayes clasifica cada flor con la especie cuya densidad (x prior) es mayor en ese punto.")

**Lectura de negocio.** El enfoque generativo **describe cada grupo** (perfil típico y dispersión), por eso funciona bien con **pocos datos** o clases muy pequeñas —justo el escenario del fraude, que es raro—. El discriminativo suele dar **mejor frontera** cuando hay muchos datos. Saber cuál conviene según el volumen evita elegir el modelo equivocado.

### LDA, QDA y Naive Bayes: los tres generativos — capítulo 10.3 (subsección 3.2)

- **LDA** modela cada clase con una gaussiana y asume que **todas comparten la misma covarianza** `Σ` → frontera **lineal**. Es la proyección de Fisher: la combinación lineal que **maximiza la separación entre clases** relativa a la variación dentro de clase.
- **QDA** deja que **cada clase tenga su covarianza** `Σ_k` → frontera **cuadrática** (curva). Más flexible, pero estima más parámetros y necesita más datos.
- **Naive Bayes** asume que las variables son **independientes dada la clase**: `P(x|clase)=∏_j P(x_j|clase)`. Casi nunca se cumple, y aun así **clasifica bien** (Domingos & Pazzani, 1997): lo que importa es **ordenar** las clases, no calibrar la probabilidad.

#### 🧮 Matemática en el cuerpo — el discriminante de Fisher y LDA (covarianza compartida) vs. QDA (por clase)

**Discriminante de Fisher (1936).** Busca la dirección $w$ que **maximiza la separación entre clases** relativa a la variación **dentro** de cada clase:

$$J(w) \;=\; \frac{w^{\top} S_B\, w}{w^{\top} S_W\, w}, \qquad w^{\*} \;\propto\; S_W^{-1}(\mu_1-\mu_0)$$

con **dispersión dentro** de clase $S_W=\sum_k \sum_{i\in k}(x_i-\mu_k)(x_i-\mu_k)^{\top}$ y **dispersión entre** clases $S_B=\sum_k n_k(\mu_k-\mu)(\mu_k-\mu)^{\top}$.

**LDA** (gaussiana por clase con **covarianza COMPARTIDA** $\Sigma_k=\Sigma$): los términos cuadráticos se cancelan y la función discriminante es **lineal** en $x$ → **frontera recta**:

$$\delta_k(x) \;=\; x^{\top}\Sigma^{-1}\mu_k \;-\; \tfrac{1}{2}\,\mu_k^{\top}\Sigma^{-1}\mu_k \;+\; \ln \pi_k$$

**QDA** (una **covarianza por clase** $\Sigma_k$): no se cancelan los cuadráticos → función **cuadrática** → **frontera curva**:

$$\delta_k(x) \;=\; -\tfrac{1}{2}\ln|\Sigma_k| \;-\; \tfrac{1}{2}(x-\mu_k)^{\top}\Sigma_k^{-1}(x-\mu_k) \;+\; \ln \pi_k$$

**La forma de la frontera ES el supuesto hecho geometría:** recta ⇔ covarianzas iguales (LDA); curva ⇔ covarianzas distintas (QDA). Fuente: el glosario de la sesión «Sección 3» y «Sección 4»; la guía de supuestos de la sesión Parte 2.2 y Parte 3.1.

**🔎 Qué hace este código.** Ajusta los **tres generativos** (LDA, QDA, GaussianNB) sobre Iris y reporta su accuracy por validación cruzada estratificada: LDA y QDA casi iguales; GaussianNB algo por debajo (viola la independencia y aun así clasifica bien).

In [ ]:
# Mini-demo: accuracy por validación cruzada de los tres generativos sobre Iris
Xd = iris_demo[FEATS].values
yd = iris_demo["species"].astype("category").cat.codes.values
cv_demo = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for nombre, clf in [("LDA", LinearDiscriminantAnalysis()),
                    ("QDA", QuadraticDiscriminantAnalysis()),
                    ("GaussianNB", GaussianNB())]:
    sc = cross_val_score(clf, Xd, yd, cv=cv_demo)
    print(f"{nombre:11s} accuracy CV = {sc.mean():.4f}  (±{sc.std():.4f})   frontera: "
          + {"LDA": "lineal", "QDA": "cuadrática", "GaussianNB": "cuadrática (ejes alineados)"}[nombre])

**Lectura de negocio.** LDA es **estable, rápido e interpretable**; se prefiere cuando iguala a QDA (menos varianza, frontera explicable). QDA se elige solo si mejora de forma **clara y consistente** (indicio de covarianzas realmente distintas entre grupos, p. ej. la variabilidad del gasto de clientes fraudulentos vs. legítimos). Naive Bayes es la **línea base ultrarrápida**, ideal con un número muy elevado de variables (texto).

#### 🖐️ Cálculo manual — la regla discriminante de LDA (covarianza combinada + función lineal)

**🔎 Qué hace este código.** Reconstruye LDA **de forma manual** sobre un subconjunto de **2 clases** de Iris (*setosa* vs. *versicolor*): estima las medias por clase y la **covarianza COMBINADA** (pooled), arma la **función discriminante lineal** $\delta_k(x)=x^{\top}\Sigma^{-1}\mu_k-\tfrac12\mu_k^{\top}\Sigma^{-1}\mu_k+\ln\pi_k$ y clasifica por el mayor $\delta_k$. Luego **verifica con `assert`** que sus predicciones y su vector de pesos $w=\Sigma^{-1}(\mu_1-\mu_0)$ coinciden con `LinearDiscriminantAnalysis`.

In [ ]:
# 🖐️ HAZLO A MANO: la regla discriminante de LDA (covarianza combinada + función discriminante lineal)
# Subconjunto de 2 clases de Iris (setosa vs versicolor) y verificación contra LinearDiscriminantAnalysis
iris_2c = obtener_iris()
sub = iris_2c[iris_2c["species"].astype(str).isin(["setosa", "versicolor"])]
X2 = sub[FEATS].values
y2 = (sub["species"].astype(str).values == "versicolor").astype(int)   # 0=setosa, 1=versicolor
n, K = len(y2), 2
mu0 = X2[y2 == 0].mean(axis=0); mu1 = X2[y2 == 1].mean(axis=0)
pi0 = (y2 == 0).mean(); pi1 = (y2 == 1).mean()

# Covarianza COMBINADA (pooled within-class), denominador n-K  -> el supuesto central de LDA
Sw = np.zeros((4, 4))
for c, mu in [(0, mu0), (1, mu1)]:
    dif = X2[y2 == c] - mu
    Sw += dif.T @ dif
Sigma = Sw / (n - K)
Sinv = np.linalg.inv(Sigma)

# Función discriminante lineal de LDA:  delta_k(x) = x . Sinv . mu_k - 0.5 mu_k . Sinv . mu_k + ln(pi_k)
def delta(x, mu, pi):
    return x @ Sinv @ mu - 0.5 * mu @ Sinv @ mu + np.log(pi)

pred_hand = np.array([1 if delta(x, mu1, pi1) > delta(x, mu0, pi0) else 0 for x in X2])
w = Sinv @ (mu1 - mu0)                                   # pesos de la frontera lineal

lda2 = LinearDiscriminantAnalysis().fit(X2, y2)
pred_sk = lda2.predict(X2)
coincide = bool((pred_hand == pred_sk).all())
cos_dir = float(np.dot(w, lda2.coef_[0]) / (np.linalg.norm(w) * np.linalg.norm(lda2.coef_[0])))

print("Pesos de la frontera lineal  w = Sigma^-1 (mu_versicolor - mu_setosa):")
print("  ", np.round(w, 3))
print(f"Predicciones a mano == LinearDiscriminantAnalysis : {coincide}  "
      f"(coinciden {int((pred_hand == pred_sk).sum())}/{n})")
print(f"Dirección de w alineada con lda.coef_ (coseno)    : {cos_dir:.4f}")
print(f"Accuracy de la regla a mano (resustitución)       : {(pred_hand == y2).mean():.4f}")

assert coincide, "la regla discriminante a mano debe reproducir las predicciones de sklearn"
assert abs(cos_dir - 1.0) < 1e-6, "w debe apuntar en la misma dirección que lda.coef_"
print("\nassert OK: la covarianza COMBINADA + la función discriminante lineal reproducen LDA.")

**📖 Cómo se lee.** La regla manual —covarianza **combinada** entre las dos clases y una **función lineal** de $x$— reproduce **exactamente** las predicciones de `LinearDiscriminantAnalysis`, y el vector de pesos $w=\Sigma^{-1}(\mu_1-\mu_0)$ apunta en la **misma dirección** que `lda.coef_` (coseno 1,0000). Ese $\Sigma$ **compartido** es lo que hace la frontera **lineal**: es el supuesto de homocedasticidad (la guía de supuestos de la sesión Parte 2.2). 💡 *setosa* y *versicolor* son linealmente separables, por eso la regla acierta el 100 % en resustitución.

### Desbalance: por qué la accuracy induce a error — capítulo 10.4 (subsección 3.3)

Cuando una clase es muy poco frecuente (fraude ≈ 0,17 %), un clasificador **trivial** que predice siempre «la mayoritaria» logra una accuracy muy alta **sin detectar nada** de la clase de interés. La accuracy alta es un **espejismo**.

**🔎 Qué hace este código.** Construye un caso 99:1 sintético y evalúa el clasificador **trivial** «todo 0»: obtiene una accuracy muy alta con **recall 0** de la clase rara. Es la **paradoja de la exactitud** en miniatura.

In [ ]:
# Mini-demo: 99:1 sintético. El clasificador "todo 0" tiene accuracy altísima y recall 0.
y_toy = np.zeros(10000, dtype=int); y_toy[:100] = 1        # 100 positivos de 10 000 (1 %)
pred_todo0 = np.zeros_like(y_toy)
acc = (pred_todo0 == y_toy).mean()
rec = recall_score(y_toy, pred_todo0)
print(f"Clasificador 'todo 0':  accuracy = {acc:.4f}  |  recall de la clase rara = {rec:.4f}")
print("99 % de acierto... y CERO positivos detectados. La accuracy oculta el fracaso.")

**Lectura de negocio.** La clase rara suele ser **la que cuesta dinero** (fraude, impago, abandono). Reportar solo accuracy en desbalance **oculta el fracaso** del modelo. Palancas de tratamiento: **SMOTE** (crea ejemplos sintéticos de la minoritaria), **undersampling** (descarta mayoritarias) y **`class_weight='balanced'`** (pondera más los errores sobre la minoritaria, sin tocar los datos).

### Métricas correctas: PR-AUC frente a ROC-AUC — capítulo 10.4 (subsección 3.4)

Sobre la clase positiva (fraude): **Precision** = VP/(VP+FP) (de los casos marcados, cuánto era fraude); **Recall** = VP/(VP+FN) (de todos los fraudes, cuántos se detectaron); **F1** = media armónica de ambas. La curva **ROC** (sensibilidad vs. FPR) da un **ROC-AUC insensible al desbalance** (se ve optimista); la curva **Precision-Recall** da el **PR-AUC (average precision)**, el **número honesto** en 99:1. La línea base del PR-AUC **no es 0,5**, sino la **prevalencia** (≈ 0,0017).

#### 🧮 Matemática en el cuerpo — precisión, recall, F1 y PR-AUC

Sobre la clase **positiva** (fraude), con verdaderos positivos $VP$, falsos positivos $FP$ y falsos negativos $FN$:

$$\text{precisión} = \frac{VP}{VP+FP}, \qquad \text{recall} = \frac{VP}{VP+FN}, \qquad F_1 = \frac{2\,\cdot\,\text{precisión}\,\cdot\,\text{recall}}{\text{precisión}+\text{recall}}$$

- **Precisión:** de las alertas, cuántas eran fraude. **Recall:** de los fraudes, cuántos se detectaron. **F1:** media armónica (penaliza el desequilibrio entre ambas).
- **PR-AUC (average precision):** área bajo la curva Precision-Recall, $\ \text{AP}=\sum_n (R_n-R_{n-1})\,P_n$. Su **línea base no es 0,5** sino la **prevalencia** de positivos.
- **ROC-AUC** es **insensible al desbalance**: la $\text{FPR}=FP/(FP+VN)$ diluye los falsos positivos en una clase negativa muy numerosa, por lo que se ve **optimista**. En 99:1 el número honesto es el **PR-AUC** (Saito & Rehmsmeier, 2015).

Fuente: el glosario de la sesión «Sección 11» y «Sección 12»; la guía de supuestos de la sesión Parte 5.2.

**🔎 Qué hace este código.** Con **los mismos scores** genera dos áreas muy distintas por el desbalance: un **ROC-AUC** alto (optimista) y un **PR-AUC** mucho menor (honesto). Imprime también la **prevalencia** = línea base del PR-AUC.

In [ ]:
# Mini-demo: mismos scores, dos áreas muy distintas por el desbalance
rng = np.random.default_rng(RANDOM_STATE)
y_ib = np.zeros(20000, dtype=int); y_ib[:60] = 1            # 60 positivos de 20 000 (0,3 %)
scores = rng.uniform(size=20000)
scores[y_ib == 1] += 1.3                                     # el modelo ordena razonablemente bien
scores = (scores - scores.min()) / (scores.max() - scores.min())
print(f"ROC-AUC = {roc_auc_score(y_ib, scores):.3f}  (alto: la FPR se diluye entre miles de negativas)")
print(f"PR-AUC  = {average_precision_score(y_ib, scores):.3f}  (mucho menor: penaliza las falsas alarmas)")
print(f"Prevalencia (línea base del PR-AUC) = {y_ib.mean():.4f}")

**Lectura de negocio.** Anunciar «AUC 0,97, excelente» sin decir que es **ROC** presenta el modelo como mejor de lo que es. En 99:1 se compara y selecciona por **PR-AUC**; el ROC-AUC se reporta solo como contexto. Además, **log-loss** penaliza estar «seguro y equivocado» y mide la **calibración** de las probabilidades, clave si la decisión pasa por un **umbral por costo** (S09).

## 10.5 — ¿Se sostiene con datos reales? Replicación del paper seminal: análisis discriminante de Fisher (1936) sobre Iris (Sección 4 del cuaderno)

### Paso 0 — Contexto del paper

**Fisher, R. A. (1936).** *The Use of Multiple Measurements in Taxonomic Problems.* **Annals of Eugenics 7(2): 179-188.** Es el artículo **fundacional del análisis discriminante lineal (LDA)**: Fisher buscó la **combinación lineal de las cuatro medidas** (largo y ancho de sépalo y pétalo) que **maximiza la separación entre especies** relativa a la variación dentro de cada especie, sobre los datos de **150 flores de iris** (50 de cada una de *setosa*, *versicolor*, *virginica*) recogidos por Edgar Anderson.

**Lo que se reproduce.** El hallazgo cualitativo de Fisher —**setosa es perfectamente separable; versicolor y virginica se solapan levemente**— y la **accuracy** de LDA/QDA por validación cruzada estratificada. QDA generaliza a LDA relajando el supuesto de covarianzas iguales.

> **Protocolo FIJADO al pie (declarado para la sesión de réplica del paper «Sección 4»).** Accuracy con `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` + `cross_val_score`; matriz de confusión y conteo de errores en un **holdout estratificado 70/30 (`random_state=42`)**. Modelos con parámetros por defecto. Cualquier desviación del protocolo cambia los decimales.

### 📄 En el paper (subsección 4.0)

**Procedencia de la réplica** (declarado para la sesión de réplica del paper «Sección 1» y «Sección 4»):

- **Análisis discriminante lineal (LDA)** — **Fisher, R. A. (1936).** *The Use of Multiple Measurements in Taxonomic Problems.* **Annals of Eugenics 7(2): 179-188.** DOI 10.1111/j.1469-1809.1936.tb02137.x. Artículo **fundacional del LDA**: la combinación lineal de las cuatro medidas que **maximiza la separación entre especies** relativa a la variación dentro de cada especie, sobre las **150 flores** de iris (50 de *setosa*, *versicolor*, *virginica*) recogidas por Edgar Anderson.
- **Lo que se reproduce.** El hallazgo cualitativo de Fisher —**setosa perfectamente separable; versicolor y virginica se solapan levemente**— y la **accuracy** de LDA/QDA por validación cruzada estratificada. **QDA** generaliza a LDA relajando el supuesto de covarianzas iguales.
- **Operativo (venv) vs. benchmark ETIQUETADO.** Valores OPERATIVOS de este cuaderno: **LDA 0,9733** y **QDA 0,9800** (`StratifiedKFold(5, shuffle, rs=42)`). Benchmark **publicado, otro pipeline**: **LDA 0,978 / QDA 0,973** con `RepeatedStratifiedKFold` 10×3 (docs scikit-learn / literatura) — cae dentro de la tolerancia ±0,03, se cita **etiquetado**, nunca como resultado propio.
- **Datasets.** Iris vía `load_iris` (offline, idéntico a UCI id 53). Caso de negocio: **Credit Card Fraud (ULB / Worldline)**, 284 807 transacciones, 492 fraudes (0,172 %); su descripción **recomienda medir con PR-AUC** por el desbalance.

### Qué preguntaba Fisher, y por qué usó lo que usó — Sección 0 del paper (subsección 4.0)

**💡 Antes de tocar los datos.** Una réplica sin esta pregunta se vuelve mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba el autor** y **por qué eligió cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El objetivo no era inventar un clasificador ni fabricar un conjunto de datos de práctica.** El principio ya estaba en uso en el entorno de Fisher: E. S. Martin lo había aplicado a las diferencias de sexo en medidas de la mandíbula y Mildred Barnard, a la tendencia secular de una serie de cráneos datados (p. 179). Lo que este artículo aporta lo declara su autor en la primera página: «In the present paper the application of the same principle will be illustrated on a taxonomic problem; some questions connected with the precision of the processes employed will also be discussed» (p. 179). Son dos preguntas superpuestas: la **taxonómica** —¿puede decidirse la especie de un ejemplar «if the specific nature were judged wholly from the measurements» (p. 182), esto es, solo con las medidas?— y la **metodológica** —¿con qué precisión, si los coeficientes se eligieron justamente para que la separación se viera lo más grande posible?—.

**Qué decisión dependía de la respuesta** (p. 185). No una clasificación de flores, sino un problema de genética: Randolph (1934) había contado los cromosomas de las tres especies —*setosa* diploide con 38, *virginica* tetraploide con 70, *versicolor* hexaploide— y Anderson sugirió que *I. versicolor* podía ser un **híbrido poliploide** de las otras dos. Si los genes actuaran de forma aditiva, versicolor debería situarse a **dos tercios del camino** entre setosa y virginica. El número que Fisher busca decide si ese origen híbrido es compatible con las medidas de las flores.

**Con qué contaba** (p. 179 y p. 185). Los datos no eran suyos: **150 flores, 50 por especie**, cuatro medidas cada una, tomadas por el botánico **Edgar Anderson** para su propio trabajo sobre el género. Fisher declara una restricción de muestreo **antes** de apoyarse en ella —virginica no procede de la misma colonia que las otras dos, «a circumstance which might considerably disturb both the mean values and their variabilities»—, que hoy se llamaría cambio de población entre muestras. Y todo el cómputo era manual: el núcleo del procedimiento consiste en invertir manualmente la matriz 4×4 de sumas de cuadrados y productos **dentro de especies**, con **98 grados de libertad** (Tablas III y IV, p. 181).

**Por qué las cuatro medidas juntas y no la mejor de las cuatro** (pp. 179 y 182). La pregunta que abre el procedimiento es literal: «What linear function of the four measurements […] will maximize the ratio of the difference between the specific means to the standard deviations within species?». El detalle de mayor valor es el **ancho de sépalo**: su diferencia entre especies es la más pequeña de las cuatro y además de **signo contrario** (−0,658 cm, Tabla II), y aun así recibe el **segundo peso más grande del compuesto, +5,9037**, frente al 1 del largo de sépalo. La medida que **peor separa por sí sola** resulta valiosa **en compañía**, porque corrige la variación conjunta de las demás — algo que ninguna comparación variable por variable puede ver.

**Por qué un cociente y una sola matriz de dispersión** (pp. 181, 182 y 186). El criterio es «the ratio D²/S», separación entre medias dividida por variación dentro de especies: maximizar solo la distancia no sirve, porque basta multiplicar por diez los coeficientes para multiplicarla por diez sin separar nada; el cociente fija una **dirección**, no una escala, y por eso Fisher normaliza manualmente —«if we choose to take the coefficient of sepal length to be unity»—. La frontera resulta **recta** porque el compuesto se apoya en una **única** matriz de dispersión dentro de especies; y Fisher no dio ese supuesto por bueno: con tres especies comprobó que las dispersiones internas **no eran iguales** y las combinó con pesos (virginica por 16, versicolor por 1, setosa por 25). Ese es el supuesto que **QDA** relaja con una covarianza por clase, y la razón de comparar ambos modelos más abajo.

**Por qué la validación cruzada del Paso 2 desciende de 1936** (pp. 185 y 187). La Sección V del artículo nombra el problema que hoy señalaría cualquier revisor: la separación observada está inflada porque la dirección se eligió **para** maximizarla, «so as to allow for the fact that a variate has been chosen so as to maximise the distinctness of the species». Su remedio fue corregir los grados de libertad (4 en la regresión, 95 en el residuo, z = 3,2183). Lo que no tenía era evaluación fuera de muestra: **todos sus números salen de las mismas plantas con las que eligió los coeficientes**. Este cuaderno añade la pieza que faltaba con `StratifiedKFold(5, shuffle=True, random_state=42)`.

**⚠️ De ahí la diferencia con el paper.** Fisher **no publicó accuracy ni matriz de confusión**: publicó un cociente de separación y una prueba de significación. Los targets de esta réplica son la lectura moderna de **su propio criterio** —«the probability of misclassification, if the specific nature were judged wholly from the measurements» (p. 182)—, no cifras copiadas del artículo. Sus coeficientes tampoco valen como target literal: el original los define **salvo escala** (p. 181) y `scikit-learn` usa otra normalización, así que solo las **proporciones** entre ellos son comparables; además Fisher ajustó un compuesto **por par** de especies y aquí se ajusta un único modelo de tres clases. Lo que sí se reproduce intacto es la conclusión: **las medidas bastan para diagnosticar setosa y no bastan para separar con certeza versicolor de virginica.**


**🔎 Qué hace este código (Paso 1).** Carga Iris con `obtener_iris()` y verifica **150×4** con **3 clases de 50**. Es la base original de Fisher (1936).

In [ ]:
# Paso 1 — Cargar Iris y verificar 150 x 4 features x 3 clases de 50
iris = obtener_iris()
print("Iris:", iris.shape, "filas x columnas")
X = iris[FEATS].values
y = iris["species"].astype("category").cat.codes.values       # 0=setosa, 1=versicolor, 2=virginica
print("Distribución de clases:", dict(zip(ESPECIES, np.bincount(y))))
display(iris.head(3))

**❓ Qué se quiere averiguar.** ¿Sigue vigente, fuera de muestra, la regla que Fisher publicó en 1936 —una frontera **recta** trazada sobre cuatro medidas de la flor—, o hace falta una frontera **curva** para separar las tres especies?

- **Qué decide:** cuál de los tres generativos se lleva al resto de la sesión y, por analogía, cuál se defiende ante un área de negocio. LDA supone que **todas las clases varían igual** y a cambio entrega una frontera lineal, de menor costo y explicable; QDA estima una covarianza por clase y solo merece ese precio si compra acierto.
- **Antes de mirar el resultado:** si QDA superara a LDA por un margen **mayor que la desviación entre pliegues** (aquí ≈ 0,03-0,04), las tres especies tendrían dispersiones lo bastante distintas como para justificar la complejidad extra. Si ambos quedan a **menos de un caso de distancia** sobre 150 flores, el supuesto de covarianza común no tiene costo y decide la simplicidad. Y si GaussianNB no pierde accuracy de forma marcada pese a suponer independencia entre medidas que, por inspección visual, se ven relacionadas, un supuesto falso tampoco impide clasificar.

**🔎 Qué hace este código (Paso 2).** Calcula la **accuracy** de LDA, QDA y GaussianNB por `StratifiedKFold(5, shuffle, rs=42)` + `cross_val_score` (targets #1-#2). El protocolo está **fijado al pie**: cualquier desviación cambia los decimales.

In [ ]:
# Paso 2 — Accuracy de LDA y QDA por validación cruzada estratificada (targets #1-#2)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lda_scores = cross_val_score(LinearDiscriminantAnalysis(), X, y, cv=cv)
qda_scores = cross_val_score(QuadraticDiscriminantAnalysis(), X, y, cv=cv)
gnb_scores = cross_val_score(GaussianNB(), X, y, cv=cv)

acc_lda = float(lda_scores.mean()); acc_qda = float(qda_scores.mean()); acc_gnb = float(gnb_scores.mean())
tabla_cv = pd.DataFrame({
    "modelo": ["LDA", "QDA", "GaussianNB (contexto)"],
    "accuracy_cv": [round(acc_lda, 4), round(acc_qda, 4), round(acc_gnb, 4)],
    "desv_folds": [round(lda_scores.std(), 4), round(qda_scores.std(), 4), round(gnb_scores.std(), 4)],
})
display(tabla_cv)
print(f"[#1] LDA accuracy (SKF5, shuffle, rs=42) = {acc_lda:.4f}   (OPERATIVO)")
print(f"[#2] QDA accuracy (mismo protocolo)      = {acc_qda:.4f}   (OPERATIVO)")
print(f"     GaussianNB accuracy (contexto)      = {acc_gnb:.4f}")
print("\nBenchmark publicado (otro pipeline, ETIQUETADO): LDA 0,978 / QDA 0,973 con "
      "RepeatedStratifiedKFold 10x3 (docs sklearn / literatura). Cae dentro de la tolerancia ±0,03.")

**Lectura de negocio.** Una accuracy de ≈ 0,98 por validación cruzada significa que ≈ 2 de cada 100 flores se clasifican mal **fuera de muestra**. LDA ≈ QDA aquí → se prefiere **LDA** (más simple, menos varianza, frontera lineal interpretable). GaussianNB queda algo por debajo pese a que largo y ancho de pétalo están correlacionados (viola la independencia): ilustra que Naive Bayes **clasifica bien igual**.

**🔎 Qué hace este código (Paso 3).** Proyecta Iris a los **2 ejes discriminantes** de LDA (setosa se separa sobre LD1) y prepara los puntos `petal_length × petal_width` para trazar después la **frontera** LDA (recta) vs. QDA (curva).

In [ ]:
# Paso 3 — Proyección discriminante de LDA (2 ejes) y datos para la frontera de decisión
proj = LinearDiscriminantAnalysis(n_components=2).fit(X, y).transform(X)
iris_proj = pd.DataFrame({"LD1": proj[:, 0], "LD2": proj[:, 1],
                          "especie": [ESPECIES[c] for c in y]})
# Datos para la frontera LDA (recta) vs QDA (curva) sobre petal length x petal width
frontera_pts = pd.DataFrame({"petal_length_cm": X[:, 2], "petal_width_cm": X[:, 3],
                             "especie": [ESPECIES[c] for c in y]})
print("Proyección LDA lista (setosa se separa por completo del resto sobre LD1).")
print("Separación entre medias de LD1:  setosa vs (versicolor, virginica) =",
      round(float(iris_proj.loc[iris_proj.especie == 'setosa', 'LD1'].mean() -
                  iris_proj.loc[iris_proj.especie != 'setosa', 'LD1'].mean()), 2))

**🔎 Qué hace este código (Paso 4).** Ajusta LDA en un **holdout estratificado 70/30 (rs=42)** y calcula su **matriz de confusión**: separa **setosa sin error** (A1) y comete muy pocos entre versicolor/virginica (A2). Reproduce a Fisher (1936).

In [ ]:
# Paso 4 — Errores de clasificación: holdout estratificado 70/30 (rs=42) + resustitución
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
lda_h = LinearDiscriminantAnalysis().fit(Xtr, ytr)
cm_iris = confusion_matrix(yte, lda_h.predict(Xte))
setosa_err = int(cm_iris[0].sum() - cm_iris[0, 0])
err_holdout = int((lda_h.predict(Xte) != yte).sum())
err_resub = int((LinearDiscriminantAnalysis().fit(X, y).predict(X) != y).sum())

print("Matriz de confusión de LDA (holdout 30 % = 45 flores):")
print(pd.DataFrame(cm_iris, index=[f"real_{e}" for e in ESPECIES],
                   columns=[f"pred_{e}" for e in ESPECIES]))
print(f"\n[A1] errores en setosa            = {setosa_err}  (esperado 0: linealmente separable)")
print(f"[A2] errores totales (holdout)    = {err_holdout} de {len(yte)}")
print(f"     errores totales (resustitución) = {err_resub} de 150")
print("\nReproduce a Fisher (1936): setosa se separa sin error; los pocos fallos están "
      "entre versicolor y virginica (se solapan).")

### Paso 5 — Comparación y lectura de negocio

**Comparación con el paper (benchmark histórico ETIQUETADO).** El resultado cualitativo de Fisher (1936) —**setosa separable; versicolor/virginica se solapan**— se reproduce: 0 errores en setosa y del orden de 1-3 errores entre las otras dos. Las accuracies operativas del venv (LDA ≈ 0,97, QDA ≈ 0,98) coinciden con la banda publicada (0,973-0,978, *otro pipeline*).

**Lectura de negocio.** La matriz de confusión **localiza** el problema: dice **entre qué clases** se equivoca el modelo (versicolor↔virginica), que es la información accionable. Si el negocio necesitara separar ese par con cero error, LDA no bastaría y habría que buscar más variables o un modelo más flexible. Naive Bayes, aun violando la independencia, queda cerca: sirve de **línea base** rápida.

## 10.8 — ¿Qué decisión habilita? Laboratorio de negocio: detección de fraude con tarjeta (ULB, desbalance 99:1) (Sección 5 del cuaderno)

**Credit Card Fraud (ULB / Worldline).** 284 807 transacciones europeas de septiembre de 2013; **492 fraudes = 0,172 %** (desbalance real ≈ 578:1, redondeado como «99:1» en el discurso). 31 columnas: `Time`, `V1…V28` (componentes PCA anonimizados), `Amount` y `Class` (1 = fraude). La descripción del dataset **recomienda medir con AUPRC (PR-AUC)** por el desbalance. Guía paso a paso en `laboratorio/GUIA_LABORATORIO_S10.docx`.

> **Se trabaja sobre el dataset COMPLETO.** El laboratorio carga el CSV completo (`cargar_fraude_completo()`), que se descarga fuera de OneDrive en runtime. La muestra local `data/creditcard_muestra.csv` (≈ 26,7 % de fraude) es **solo para inspección offline** y **no** se usa para los resultados: el desbalance 99:1 solo es real en el completo.

**🔎 Qué hace este código (Paso 6).** Descarga (en runtime, fuera de OneDrive) y carga el **fraude COMPLETO** y verifica **284 807×31** con **492 fraudes** (0,172 %) → desbalance ≈ 578:1.

In [ ]:
# Paso 6 — Cargar el fraude COMPLETO y verificar 284807 x 31 con 492 fraudes
fraude = cargar_fraude_completo()
print("Fraude (COMPLETO):", fraude.shape)
Xf = fraude.drop(columns="Class").values
yf = fraude["Class"].astype(int).values
n_pos, n_tot = int(yf.sum()), len(yf)
print(f"Fraudes = {n_pos} de {n_tot}  ({100 * n_pos / n_tot:.3f} %)  ->  desbalance ~{n_tot // n_pos}:1")
print("Columnas:", list(fraude.columns)[:6], "...", list(fraude.columns)[-2:])

**❓ Qué se quiere averiguar.** ¿Cuánto «acierto» puede exhibir un sistema antifraude que **no detecta ni un solo fraude**?

- **Qué decide:** el piso por debajo del cual la palabra acierto deja de significar algo en este problema. Si un comité aprueba modelos por porcentaje de acierto, esa cifra es la primera que debe conocer: marca cuánto vale «no hacer nada».
- **Antes de mirar el resultado:** hay **492 fraudes entre 284 807** transacciones. Si la accuracy del trivial saliera moderada (del orden de 0,60), la métrica todavía discriminaría entre un modelo y otro. Si resulta **superior a 0,99**, entonces cualquier presentación que anuncie «99 % de acierto» equivale al sistema que no revisa nada, y su recall de fraude será **cero**.

**🔎 Qué hace este código (Paso 7, drill 1).** Construye el clasificador **trivial** «todo legítimo» sobre el fraude completo: accuracy = 1 − 492/284 807 = **0,9983** y **recall de fraude = 0** (target #3). Muestra la matriz de confusión **degenerada**.

In [ ]:
# Paso 7 — Paradoja de la accuracy (drill 1): clasificador trivial "todo legítimo"
pred_trivial = np.zeros_like(yf)
acc_trivial = float((pred_trivial == yf).mean())
recall_trivial = float(recall_score(yf, pred_trivial))       # 0 verdaderos positivos -> 0
cm_trivial = confusion_matrix(yf, pred_trivial)
print(f"[#3] Clasificador 'todo legítimo':  accuracy = {acc_trivial:.4f}  |  "
      f"recall de fraude = {recall_trivial:.3f}")
print("Matriz de confusión degenerada (la columna de positivos predichos queda vacía):")
print(pd.DataFrame(cm_trivial, index=["real_legítima", "real_fraude"],
                   columns=["pred_legítima", "pred_fraude"]))
print("\nUn 99,83 % de acierto SIN detectar un solo fraude. La accuracy no mide nada útil aquí.")

**Lectura de negocio (drill 1).** El 0,9983 es un espejismo: se obtiene ignorando por completo la clase que cuesta dinero. **Nunca** se elige ni se reporta un modelo por accuracy en desbalance; se fijan de antemano las métricas de la clase rara (recall, precision, F1, PR-AUC). La precision del trivial es **indefinida** (0/0), no 0.

**❓ Qué se quiere averiguar.** De cada 100 transacciones que el sistema marque como sospechosas, ¿cuántas serán realmente fraude? Y ¿por qué **dos áreas bajo curva calculadas con los mismos scores** no contestan lo mismo?

- **Qué decide:** el número que se lleva al comité y, con él, cuántas revisiones manuales pagará la empresa por cada fraude detectado.
- **Antes de mirar el resultado:** la línea base del PR-AUC **no es 0,5 sino la prevalencia** (≈ 0,0017). Si ROC-AUC y PR-AUC salieran parecidos, cualquiera de los dos serviría para elegir modelo. Si el ROC-AUC ronda 0,96 y el PR-AUC queda claramente por debajo, la **brecha** entre ambos mide el optimismo que aporta el desbalance, y elegir por ROC sería elegir por la métrica más halagadora.

**🔎 Qué hace este código (Paso 8, drill 2).** Split estratificado 70/30, `StandardScaler` **solo en train** (sin fuga) y **logística base**. Calcula **ROC-AUC** (A3 ≈ 0,957) y **PR-AUC** (target #4 ≈ 0,708), y recall/precisión en umbral 0,5. La brecha ROC ≫ PR es el síntoma del desbalance. La logística se ajusta con `max_iter=1000`: eleva el tope de iteraciones del solver `lbfgs` (por defecto 100) para que la optimización **converja** sobre las 30 variables escaladas y no dispare el aviso de no-convergencia; no altera las cifras, solo garantiza que el ajuste termina.

In [ ]:
# Paso 8 — Clasificador base + ROC vs PR (drill 2). Split 70/30 estratificado, escalado SOLO en train.
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.30, stratify=yf, random_state=42)
escalador = StandardScaler().fit(Xtr)                        # se ajusta SOLO con el train (sin fuga)
Xtr_s, Xte_s = escalador.transform(Xtr), escalador.transform(Xte)

base = LogisticRegression(max_iter=1000).fit(Xtr_s, ytr)
proba_base = base.predict_proba(Xte_s)[:, 1]
roc_auc_base = float(roc_auc_score(yte, proba_base))
pr_auc_base = float(average_precision_score(yte, proba_base))
fpr, tpr, _ = roc_curve(yte, proba_base)
prec_c, rec_c, _ = precision_recall_curve(yte, proba_base)

pred_base = (proba_base >= 0.5).astype(int)
recall_base = float(recall_score(yte, pred_base))
prec_base = float(precision_score(yte, pred_base, zero_division=0))
cm_base = confusion_matrix(yte, pred_base)
print(f"[A3] ROC-AUC base = {roc_auc_base:.4f}   (alto: la FPR se diluye entre ~85 000 legítimas)")
print(f"[#4] PR-AUC base  = {pr_auc_base:.4f}   (mucho menor: el número HONESTO en 99:1)   OPERATIVO")
print(f"     recall (fraude, umbral 0,5) = {recall_base:.4f}  |  precision = {prec_base:.4f}")
print("\nBenchmark publicado (ETIQUETADO, otro pipeline): ROC ≈ 0,97 / PR-AUC ≈ 0,70-0,78 (literatura).")

**Lectura de negocio (drill 2).** El ROC-AUC ≈ 0,96 parece excelente, pero induce a error: un salto de cientos de falsos positivos apenas mueve la FPR entre ~85 000 legítimas. El **PR-AUC ≈ 0,71** es el número honesto —refleja cuántas de las alertas son fraude real— y muestra que el modelo tiene señal pero está lejos de ser perfecto. La **brecha ROC ≫ PR es el síntoma** del desbalance (Saito & Rehmsmeier, 2015).

#### 🖐️ Cálculo manual — precisión, recall y PR-AUC (average precision) del fraude base

**🔎 Qué hace este código.** Toma la matriz de confusión del clasificador base (Paso 8) y recompone **manualmente** precisión = $VP/(VP+FP)$, recall = $VP/(VP+FN)$ y $F_1$; y además reconstruye la **métrica insignia del desbalance, el PR-AUC (average precision)**, como **suma de rectángulos** sobre la curva precisión-recall —$AP=\sum_n (R_n-R_{n-1})\,P_n$—. **Verifica con `assert`** que las tres igualan a `precision_score`/`recall_score`/`average_precision_score` de sklearn: la métrica que **ordena el benchmark** queda reproducible desde el material.

In [ ]:
# 🖐️ HAZLO A MANO: precisión, recall y PR-AUC (average precision) del fraude base == sklearn
VN, FP, FN, VP = cm_base.ravel()          # orden de sklearn: [[VN, FP], [FN, VP]]  (cm_base viene del Paso 8)
precision_mano = VP / (VP + FP)           # de las alertas, cuántas eran fraude
recall_mano = VP / (VP + FN)              # de los fraudes, cuántos se detectaron
f1_mano = 2 * precision_mano * recall_mano / (precision_mano + recall_mano)

print(f"Matriz de confusión base:  VP={VP}  FP={FP}  FN={FN}  VN={VN}")
print(f"  precisión = VP/(VP+FP) = {VP}/{VP + FP} = {precision_mano:.4f}")
print(f"  recall    = VP/(VP+FN) = {VP}/{VP + FN} = {recall_mano:.4f}")
print(f"  F1        = 2PR/(P+R)                 = {f1_mano:.4f}")

assert abs(precision_mano - prec_base) < 1e-9, "la precisión a mano debe igualar a sklearn"
assert abs(recall_mano - recall_base) < 1e-9, "el recall a mano debe igualar a sklearn"
print(f"assert OK: a mano == sklearn (precision_score {prec_base:.4f}, recall_score {recall_base:.4f}).")

# --- PR-AUC (average precision) A MANO: la métrica INSIGNIA del desbalance, por SUMA DE RECTÁNGULOS ---
# average_precision integra la curva precisión-recall como función ESCALÓN (no trapecios): recorre los
# umbrales de mayor a menor score y acumula, en cada escalón, el ANCHO del salto de recall por la
# ALTURA (precisión) de ese punto -> AP = Σ_n (R_n − R_(n−1))·P_n, con la convención P(R=0)=1.
prec_curva, rec_curva, _ = precision_recall_curve(yte, proba_base)          # curva completa (proba_base = Paso 8)
ap_mano = float(np.sum((rec_curva[:-1] - rec_curva[1:]) * prec_curva[:-1]))  # suma de ancho·altura de cada rectángulo
print(f"\nPR-AUC (average precision) a mano = suma de rectángulos = {ap_mano:.4f}   (métrica insignia; target #4)")
assert abs(ap_mano - pr_auc_base) < 1e-9, "el AP a mano debe igualar a average_precision_score"
print(f"assert OK: AP a mano == average_precision_score ({pr_auc_base:.4f}) — la métrica insignia es reproducible.")

**📖 Cómo se lee.** Con **VP 91, FP 16, FN 57**, la precisión (91/107 = **0,8505**) dice que la mayoría de las alertas eran fraude, pero el recall (91/148 = **0,6149**) revela que **no se detecta ~38 % de los fraudes**. ⚠️ En desbalance hay que **nombrar la clase**: estas métricas son de la clase **positiva** (fraude); promediarlas ocultaría el fracaso (la guía de supuestos de la sesión Parte 5.2). El **PR-AUC calculado manualmente** (0,708) coincide con `average_precision_score` al sumar los **rectángulos** de la curva —recorrer los umbrales y acumular, en cada salto de recall, la precisión de ese punto—: así la métrica que **ordena el benchmark** no es una caja negra, sino aritmética reproducible.

**❓ Qué se quiere averiguar.** ¿Cuántos fraudes más detecta el modelo si se le entrena con un train **equilibrado** por SMOTE, y a cuántas falsas alarmas obliga esa ganancia?

- **Qué decide:** el tamaño del equipo de revisión. Cada falsa alarma es una verificación manual pagada y una tarjeta legítima bloqueada; cada fraude no detectado es una pérdida directa. La comparación entre ambos costos —no el algoritmo— es la que hace que SMOTE convenga o no.
- **Antes de mirar el resultado:** en el umbral 0,5 el modelo base detecta **91 de 148** fraudes con **16** falsas alarmas y una precisión de **0,85**. Si tras SMOTE el recall sube y la precisión apenas se mueve, el remuestreo es una mejora de costo casi nulo. Si el recall sube y la precisión cae de forma marcada por debajo de 0,10, lo obtenido no es un modelo mejor sino un modelo **más alarmista**: la ganancia existe, pero su costo recae en el equipo que revisa.

**🔎 Qué hace este código (Paso 9, drill 3).** Aplica `SMOTE(random_state=42)` **solo al train** (nunca antes del split: sería fuga), reentrena la logística y compara **recall** y **precisión** de fraude en el **test real**: el recall **sube** (target #5), la precisión **baja**.

In [ ]:
# Paso 9 — Efecto de SMOTE (drill 3). SMOTE SOLO en el train (nunca antes del split: evita leakage).
Xr, yr = SMOTE(random_state=42).fit_resample(Xtr_s, ytr)     # remuestreo del train escalado
print(f"Train original: {np.bincount(ytr)}  ->  tras SMOTE: {np.bincount(yr)}  (clases equilibradas)")

clf_sm = LogisticRegression(max_iter=1000).fit(Xr, yr)
proba_sm = clf_sm.predict_proba(Xte_s)[:, 1]                 # se evalúa en el test REAL (sin remuestrear)
pred_sm = (proba_sm >= 0.5).astype(int)
recall_sm = float(recall_score(yte, pred_sm))
prec_sm = float(precision_score(yte, pred_sm, zero_division=0))
cm_sm = confusion_matrix(yte, pred_sm)
print(f"[#5] recall de fraude:  base = {recall_base:.4f}  ->  post-SMOTE = {recall_sm:.4f}  "
      f"(SUBE, Δ = {recall_sm - recall_base:+.4f})   OPERATIVO")
print(f"[A4] precision de fraude: base = {prec_base:.4f}  ->  post-SMOTE = {prec_sm:.4f}  (BAJA)")
print("\nAlternativas al remuestreo sintético: `class_weight='balanced'` (pondera la minoritaria sin "
      "tocar los datos) y undersampling (descarta mayoritarias). Se comparan en el benchmark.")

> **Advertencia crítica (leakage).** SMOTE debe aplicarse **solo al conjunto de entrenamiento**, **después** del split. Aplicarlo antes (o sobre todo el dataset) **filtra información del test** e **infla** todas las métricas: Hayat & Magnier (2025, arXiv 2506.02703) muestran que una red mínima con ese fallo «alcanza 99,9 % de recall» de forma engañosa. El test **conserva** la distribución real (99,83 % : 0,17 %).

**Lectura de negocio (drill 3).** SMOTE vuelve al modelo **más sensible**: detecta más fraudes (recall↑) al precio de más falsas alarmas (precision↓). Es un **trade-off**, no una mejora sin costo. Conviene si el costo de un fraude no detectado domina y el equipo puede absorber más alertas; si la precisión cae demasiado (muchas tarjetas legítimas bloqueadas), se prefiere `class_weight` o ajustar el **umbral** (S09).

**❓ Qué se quiere averiguar.** Con seis candidatos sobre la mesa, ¿cuál se recomienda para producción, y con qué evidencia se sostiene esa recomendación ante quien paga las revisiones?

- **Qué decide:** el modelo del entregable. Es la elección que hay que defender con una tabla y un protocolo común, no con una preferencia.
- **Antes de mirar el resultado:** si los tratamientos del desbalance (SMOTE, `class_weight`) encabezaran el PR-AUC, reequilibrar mejoraría realmente la capacidad de **ordenar** transacciones por riesgo. Si el campeón por PR-AUC resulta ser la logística **sin tratar**, entonces esos tratamientos no ordenan mejor: solo desplazan el punto de corte. Conviene además comprobar que el ganador **cambia según la columna** por la que se ordene la tabla —por recall lidera otro—: la métrica elegida decide tanto como los datos.

**🔎 Qué hace este código (Paso 10, entregable).** Construye el **benchmark reproducible**: LDA, QDA, GaussianNB, logística base, logística+SMOTE y logística con `class_weight`, todos con las **métricas correctas** (precision/recall/F1/PR-AUC/log-loss, **no** accuracy). Elige el **campeón por PR-AUC**.

In [ ]:
# Paso 10 — Benchmark reproducible (entregable). Métricas correctas, NO accuracy. Campeón por PR-AUC.
def evaluar(nombre, clf, Xtr_, ytr_):
    clf.fit(Xtr_, ytr_)
    p = clf.predict_proba(Xte_s)[:, 1]
    yp = (p >= 0.5).astype(int)
    return {"modelo": nombre,
            "precision": round(float(precision_score(yte, yp, zero_division=0)), 4),
            "recall": round(float(recall_score(yte, yp)), 4),
            "f1": round(float(f1_score(yte, yp)), 4),
            "pr_auc": round(float(average_precision_score(yte, p)), 4),
            "roc_auc": round(float(roc_auc_score(yte, p)), 4),
            "log_loss": round(float(log_loss(yte, clf.predict_proba(Xte_s))), 4)}

filas = [
    evaluar("LogReg (base)", LogisticRegression(max_iter=1000), Xtr_s, ytr),
    evaluar("LDA", LinearDiscriminantAnalysis(), Xtr_s, ytr),
    evaluar("QDA", QuadraticDiscriminantAnalysis(reg_param=0.01), Xtr_s, ytr),
    evaluar("GaussianNB", GaussianNB(), Xtr_s, ytr),
    evaluar("LogReg + SMOTE", LogisticRegression(max_iter=1000), Xr, yr),
    evaluar("LogReg (class_weight)", LogisticRegression(max_iter=1000, class_weight="balanced"), Xtr_s, ytr),
]
benchmark = pd.DataFrame(filas)[["modelo", "precision", "recall", "f1", "pr_auc", "roc_auc", "log_loss"]]
display(benchmark)

campeon = benchmark.loc[benchmark["pr_auc"].idxmax(), "modelo"]
print(f"Modelo campeón por PR-AUC = {campeon}  (PR-AUC = {benchmark['pr_auc'].max():.4f})")
print("Se ordena por PR-AUC (métrica correcta en desbalance), NUNCA por accuracy.")
print("Umbral (reengancha S09): el campeón fija su punto de operación por COSTO —bajar el umbral sube el "
      "recall a costa de precisión; SMOTE/class_weight desplazan ese trade-off.")

**Lectura del benchmark (entregable).** La tabla compara los modelos bajo el **mismo protocolo** (split estratificado, semilla 42, escalado en train) con las **métricas correctas**. El **campeón se elige por PR-AUC**, no por accuracy ni por ROC-AUC (ambos inducen a error aquí). QDA y GaussianNB logran **recall** alto pero **precisión** muy baja (un volumen elevado de falsas alarmas → PR-AUC bajo): buenos «detectores», malos «filtros». La logística ofrece el mejor **PR-AUC**; **SMOTE** y **`class_weight`** son las palancas para subir el **recall** cuando el costo del fraude no detectado domina. La recomendación final fija el **umbral** por costo (S09) y se documenta para ser **reproducible**.

## Transversal — Exportación a Excel y figuras de resultados (Sección 6 del cuaderno)

Convención del curso: los resultados y pruebas se vuelcan a `resultados/S10_resultados.xlsx` (hoja `clasificadores_iris`, contrato `B2:B6`), y las **figuras de resultados se generan LEYENDO ese Excel**, no desde objetos en memoria. Las figuras de **EDA** (crudo) se trazan directamente. el material de referencia de la sesión lee las celdas `B2:B6`.

**🔎 Qué hace este código.** **ESCRIBE** `resultados/S10_resultados.xlsx`: la hoja de **contrato** `clasificadores_iris` (B2:B6 = targets #1-#5) más las hojas de apoyo (proyección LDA, puntos de frontera, matrices de confusión, curvas ROC/PR y benchmark) que **alimentan las figuras**. Es la **única** celda que escribe el Excel.

In [ ]:
# Construir el Excel: hoja de contrato clasificadores_iris (B2:B6) + hojas de apoyo para las figuras
from openpyxl import Workbook
wb = Workbook()

# --- Hoja 1: clasificadores_iris (CONTRATO; valores OPERATIVOS del venv) ---
ws = wb.active; ws.title = "clasificadores_iris"
ws["A1"] = "metrica"; ws["B1"] = "valor"
ws["A2"] = "accuracy_lda_iris";     ws["B2"] = round(acc_lda, 4)        # target #1
ws["A3"] = "accuracy_qda_iris";     ws["B3"] = round(acc_qda, 4)        # target #2
ws["A4"] = "recall_trivial_fraude"; ws["B4"] = round(recall_trivial, 4) # target #3
ws["A5"] = "pr_auc_base_fraude";    ws["B5"] = round(pr_auc_base, 4)    # target #4
ws["A6"] = "recall_smote_fraude";   ws["B6"] = round(recall_sm, 4)      # target #5
# Filas de apoyo (A1-A4 de la investigación)
ws["A7"]  = "accuracy_gnb_iris";      ws["B7"]  = round(acc_gnb, 4)
ws["A8"]  = "roc_auc_base_fraude";    ws["B8"]  = round(roc_auc_base, 4)
ws["A9"]  = "recall_base_fraude";     ws["B9"]  = round(recall_base, 4)
ws["A10"] = "precision_base_fraude";  ws["B10"] = round(prec_base, 4)
ws["A11"] = "precision_smote_fraude"; ws["B11"] = round(prec_sm, 4)
ws["A12"] = "errores_setosa_holdout"; ws["B12"] = setosa_err
ws["A13"] = "errores_totales_holdout";ws["B13"] = err_holdout
ws["A14"] = "acc_trivial_fraude";     ws["B14"] = round(acc_trivial, 4)

# --- Hoja 2: iris_proyeccion_lda (LD1, LD2, especie) ---
ws2 = wb.create_sheet("iris_proyeccion_lda")
ws2.append(["LD1", "LD2", "especie"])
for _, r in iris_proj.iterrows():
    ws2.append([round(float(r["LD1"]), 5), round(float(r["LD2"]), 5), r["especie"]])

# --- Hoja 3: iris_frontera_puntos (petal length x width para la frontera) ---
ws3 = wb.create_sheet("iris_frontera_puntos")
ws3.append(["petal_length_cm", "petal_width_cm", "especie"])
for _, r in frontera_pts.iterrows():
    ws3.append([float(r["petal_length_cm"]), float(r["petal_width_cm"]), r["especie"]])

# --- Hoja 4: iris_confusion_lda (matriz 3x3 del holdout) ---
ws4 = wb.create_sheet("iris_confusion_lda")
ws4.append([""] + [f"pred_{e}" for e in ESPECIES])
for i, e in enumerate(ESPECIES):
    ws4.append([f"real_{e}"] + [int(v) for v in cm_iris[i]])

# --- Hoja 5: roc_fraude (puntos de la curva ROC) ---
ws5 = wb.create_sheet("roc_fraude")
ws5.append(["fpr", "tpr"])
for a, b in zip(fpr, tpr):
    ws5.append([round(float(a), 6), round(float(b), 6)])

# --- Hoja 6: pr_fraude (puntos de la curva Precision-Recall) ---
ws6 = wb.create_sheet("pr_fraude")
ws6.append(["recall", "precision"])
for rr, pp in zip(rec_c, prec_c):
    ws6.append([round(float(rr), 6), round(float(pp), 6)])
ws6.append([])
ws6.append(["prevalencia_test", round(float(yte.mean()), 6)])

# --- Hoja 7: fraude_confusion (base y post-SMOTE, umbral 0,5) ---
ws7 = wb.create_sheet("fraude_confusion")
tn, fp, fn, tp = cm_base.ravel()
ws7.append(["base_umbral_0.5", "pred_legítima", "pred_fraude"])
ws7.append(["real_legítima", int(tn), int(fp)])
ws7.append(["real_fraude", int(fn), int(tp)])
ws7.append([])
tn2, fp2, fn2, tp2 = cm_sm.ravel()
ws7.append(["post_SMOTE_umbral_0.5", "pred_legítima", "pred_fraude"])
ws7.append(["real_legítima", int(tn2), int(fp2)])
ws7.append(["real_fraude", int(fn2), int(tp2)])

# --- Hoja 8: benchmark (tabla del entregable) ---
ws8 = wb.create_sheet("benchmark")
ws8.append(list(benchmark.columns))
for _, r in benchmark.iterrows():
    ws8.append([r["modelo"], float(r["precision"]), float(r["recall"]), float(r["f1"]),
                float(r["pr_auc"]), float(r["roc_auc"]), float(r["log_loss"])])

wb.save(XLSX)
print("Excel guardado en:", XLSX)
print("Hojas:", wb.sheetnames)
print(f"\nContrato clasificadores_iris (B2:B6):  B2(LDA)={round(acc_lda,4)}  B3(QDA)={round(acc_qda,4)}  "
      f"B4(recall trivial)={round(recall_trivial,4)}  B5(PR-AUC base)={round(pr_auc_base,4)}  "
      f"B6(recall SMOTE)={round(recall_sm,4)}")

### Figuras de resultados (leyendo el Excel)

**🔎 Qué hace este código (Figura 1).** Lee `iris_proyeccion_lda` del Excel y traza la **proyección de Fisher**: *setosa* queda completamente separada sobre LD1.

In [ ]:
# Figura 1 — Proyección discriminante de LDA (leída de iris_proyeccion_lda): setosa se separa
proj_x = pd.read_excel(XLSX, sheet_name="iris_proyeccion_lda")
fig, ax = plt.subplots(figsize=(6.2, 4.4))
for i, esp in enumerate(ESPECIES):
    sub = proj_x[proj_x["especie"] == esp]
    ax.scatter(sub["LD1"], sub["LD2"], s=32, color=PALETA[i], alpha=0.85, label=esp, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Primer eje discriminante (LD1)")
ax.set_ylabel("Segundo eje discriminante (LD2)")
ax.set_title("Proyección de Fisher (LDA): setosa queda completamente separada")
ax.legend(frameon=False, title="Especie")
mostrar(fig, FIGURAS / "iris_proyeccion_lda.png")

**🔎 Qué hace este código (Figura 2).** Traza la **frontera de decisión** de LDA (recta) y de QDA (curva) sobre `petal_length × petal_width`: la forma de la frontera revela el supuesto de covarianza.

In [ ]:
# Figura 2 — Frontera de decisión LDA (recta) vs QDA (curva) sobre petal length x width
pts = pd.read_excel(XLSX, sheet_name="iris_frontera_puntos")
cod = {e: i for i, e in enumerate(ESPECIES)}
Xg = pts[["petal_length_cm", "petal_width_cm"]].values
yg = pts["especie"].map(cod).values
x_min, x_max = Xg[:, 0].min() - 0.5, Xg[:, 0].max() + 0.5
y_min, y_max = Xg[:, 1].min() - 0.5, Xg[:, 1].max() + 0.5
xx, yy_ = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
malla = np.c_[xx.ravel(), yy_.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
for ax, (nombre, modelo) in zip(axes, [("LDA (frontera lineal)", LinearDiscriminantAnalysis()),
                                       ("QDA (frontera cuadrática)", QuadraticDiscriminantAnalysis())]):
    modelo.fit(Xg, yg)
    Z = modelo.predict(malla).reshape(xx.shape)
    ax.contourf(xx, yy_, Z, alpha=0.18, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=[PALETA[0], PALETA[1], PALETA[2]])
    for i, esp in enumerate(ESPECIES):
        sub = Xg[yg == i]
        ax.scatter(sub[:, 0], sub[:, 1], s=26, color=PALETA[i], edgecolor="white", linewidth=0.4, label=esp)
    ax.set_title(nombre)
    ax.set_xlabel("Largo del pétalo (cm)")
axes[0].set_ylabel("Ancho del pétalo (cm)")
axes[1].legend(frameon=False, fontsize=9, title="Especie")
fig.suptitle("LDA traza una frontera recta; QDA la curva (covarianzas distintas por clase)", y=1.02)
mostrar(fig, FIGURAS / "iris_fronteras_lda_qda.png")

**🔎 Qué hace este código (Figura 3).** Lee `iris_confusion_lda` del Excel y dibuja la **matriz de confusión** de LDA (holdout): localiza el único error (virginica↔versicolor).

In [ ]:
# Figura 3 — Matriz de confusión de LDA sobre Iris (holdout), leída de iris_confusion_lda
cmx = pd.read_excel(XLSX, sheet_name="iris_confusion_lda", index_col=0)
M = cmx.values.astype(int)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(M, cmap="Reds")
ax.set_xticks(range(3)); ax.set_xticklabels([f"pred\n{e}" for e in ESPECIES])
ax.set_yticks(range(3)); ax.set_yticklabels([f"real {e}" for e in ESPECIES])
for i in range(3):
    for j in range(3):
        ax.text(j, i, M[i, j], ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if M[i, j] > M.max() / 2 else UPC_TINTA)
ax.set_title("Iris — matriz de confusión de LDA (holdout 30 %)")
ax.grid(False)
mostrar(fig, FIGURAS / "iris_confusion_lda.png")

**🔎 Qué hace este código (Figura 4).** Lee `roc_fraude` del Excel y traza la **curva ROC** del fraude (se ve **optimista** por el desbalance).

In [ ]:
# Figura 4 — Curva ROC del fraude (leída de roc_fraude; ROC-AUC de clasificadores_iris)
roc = pd.read_excel(XLSX, sheet_name="roc_fraude").dropna()
meta = pd.read_excel(XLSX, sheet_name="clasificadores_iris")
roc_auc_leido = float(meta.loc[meta["metrica"] == "roc_auc_base_fraude", "valor"].iloc[0])
fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.plot(roc["fpr"], roc["tpr"], color=UPC_ROJO, lw=2.2, label=f"Logística (ROC-AUC = {roc_auc_leido:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color=UPC_GRIS, lw=1, label="Azar (0,50)")
ax.set_xlabel("Tasa de falsos positivos (1 − especificidad)")
ax.set_ylabel("Sensibilidad (recall)")
ax.set_title("Curva ROC — fraude (se ve OPTIMISTA por el desbalance)")
ax.legend(loc="lower right", frameon=False)
mostrar(fig, FIGURAS / "fraude_roc.png")

**🔎 Qué hace este código (Figura 5).** Lee `pr_fraude` del Excel y traza la **curva Precision-Recall** con su línea base = prevalencia (el número **honesto** en 99:1).

In [ ]:
# Figura 5 — Curva Precision-Recall del fraude (leída de pr_fraude; PR-AUC de clasificadores_iris)
pr = pd.read_excel(XLSX, sheet_name="pr_fraude")
prevalencia = float(pr.loc[pr["recall"] == "prevalencia_test", "precision"].iloc[0]) if \
    (pr["recall"] == "prevalencia_test").any() else float(yte.mean())
pr_curva = pr[pd.to_numeric(pr["recall"], errors="coerce").notna()].astype({"recall": float, "precision": float})
pr_auc_leido = float(meta.loc[meta["metrica"] == "pr_auc_base_fraude", "valor"].iloc[0])
fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.plot(pr_curva["recall"], pr_curva["precision"], color=UPC_ROJO, lw=2.2,
        label=f"Logística (PR-AUC = {pr_auc_leido:.3f})")
ax.axhline(prevalencia, ls="--", color=UPC_GRIS, lw=1, label=f"Línea base = prevalencia ({prevalencia:.4f})")
ax.set_xlabel("Recall (sensibilidad)")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall — el número HONESTO en 99:1")
ax.legend(loc="upper right", frameon=False)
mostrar(fig, FIGURAS / "fraude_pr.png")

**🔎 Qué hace este código (Figura 6).** Lee `fraude_confusion` del Excel y compara las matrices **base vs. post-SMOTE**: los VP suben (recall↑) y los FP aumentan de forma pronunciada (precisión↓).

In [ ]:
# Figura 6 — Matrices de confusión del fraude: base vs post-SMOTE (leídas de fraude_confusion)
crudo = pd.read_excel(XLSX, sheet_name="fraude_confusion", header=None)
def leer_bloque(fila0):
    m = crudo.iloc[fila0 + 1:fila0 + 3, 1:3].values.astype(int)
    return m
cm_base_x = leer_bloque(0)
idx_sm = crudo.index[crudo[0] == "post_SMOTE_umbral_0.5"][0]
cm_sm_x = leer_bloque(idx_sm)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.2))
for ax, M, titulo in zip(axes, [cm_base_x, cm_sm_x],
                         ["Base (umbral 0,5)", "Post-SMOTE (umbral 0,5)"]):
    im = ax.imshow(M, cmap="Reds")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["pred legítima", "pred fraude"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["real legítima", "real fraude"])
    etiquetas = np.array([["VN", "FP"], ["FN", "VP"]])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{etiquetas[i, j]}\n{M[i, j]}", ha="center", va="center", fontsize=11,
                    fontweight="bold", color="white" if M[i, j] > M.max() / 2 else UPC_TINTA)
    ax.set_title(titulo); ax.grid(False)
fig.suptitle("SMOTE sube los VP (recall↑) pero dispara los FP (precisión↓)", y=1.03)
mostrar(fig, FIGURAS / "fraude_confusion_base_smote.png")

**🔎 Qué hace este código (Figura 7).** Lee `benchmark` del Excel y ordena los modelos por **PR-AUC** (rojo = campeón): la métrica correcta en desbalance.

In [ ]:
# Figura 7 — Benchmark de clasificadores por PR-AUC (leída de benchmark)
bx = pd.read_excel(XLSX, sheet_name="benchmark").sort_values("pr_auc")
colores = [UPC_ROJO if v == bx["pr_auc"].max() else UPC_GRIS for v in bx["pr_auc"]]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.barh(bx["modelo"], bx["pr_auc"], color=colores)
for i, v in enumerate(bx["pr_auc"]):
    ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=10)
ax.set_xlabel("PR-AUC (average precision) — métrica correcta en desbalance")
ax.set_title("Benchmark de clasificadores sobre fraude (rojo = campeón por PR-AUC)")
ax.set_xlim(0, max(0.85, bx["pr_auc"].max() + 0.12))
ax.grid(axis="y")
mostrar(fig, FIGURAS / "benchmark_pr_auc.png")

### Figura de EDA (cálculo directo sobre los datos crudos)

**🔎 Qué hace este código (Figura 8, EDA).** Traza el **desbalance real** del fraude en escala logarítmica directamente sobre los datos crudos (EDA, no del Excel de resultados): 492 fraudes entre 284 807.

In [ ]:
# Figura 8 (EDA) — El desbalance real del fraude (escala logarítmica)
conteo = np.bincount(yf)
fig, ax = plt.subplots(figsize=(5.4, 4.0))
barras = ax.bar(["Legítimas (0)", "Fraude (1)"], conteo, color=[UPC_GRIS, UPC_ROJO])
ax.set_yscale("log")
for b, v in zip(barras, conteo):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v:,}\n({v / conteo.sum() * 100:.3f} %)",
            ha="center", fontsize=10)
ax.set_ylabel("Número de transacciones (escala log)")
ax.set_title("Desbalance extremo: 492 fraudes entre 284 807 transacciones (0,17 %)")
ax.grid(axis="x")
mostrar(fig, FIGURAS / "fraude_desbalance.png")

## Transversal — ✅ Verificación desde la base (Sección 6.1 del cuaderno)

El Excel de contrato ya está escrito. Ahora se comprueba que **es producto de ejecutar el código sobre los datos**, no un registro independiente del cálculo: se **recomputa** lo clave —accuracy de LDA sobre Iris y recall de fraude base y post-SMOTE— desde los datos ya cargados y se **cruza con el Excel** mediante `assert`. Refleja lo que hace el material de referencia de la sesión (que recomputa desde la base). **No toca el Excel.**

**🔎 Qué hace este código.** (1) Recomputa la **accuracy de LDA** sobre Iris fresca (`StratifiedKFold(5, shuffle, rs=42)`); (2) recompone el **recall base** y el **recall post-SMOTE** desde las matrices de confusión del fraude (`VP/(VP+FN)`); y **cruza** las tres magnitudes con la hoja `clasificadores_iris` mediante `assert`.

In [ ]:
# ✅ Verificación desde la base: recomputa lo clave desde los datos y cruza con el Excel (assert). NO toca el Excel.
meta_v = pd.read_excel(XLSX, sheet_name="clasificadores_iris")
val = lambda k: float(meta_v.loc[meta_v["metrica"] == k, "valor"].iloc[0])

# (1) accuracy de LDA sobre Iris recomputada fresca (mismo protocolo que la réplica)
cv_v = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_lda_re = float(cross_val_score(LinearDiscriminantAnalysis(), X, y, cv=cv_v).mean())
# (2)/(3) recall base y post-SMOTE recomputados desde las matrices de confusión del fraude
recall_base_re = cm_base[1, 1] / (cm_base[1, 1] + cm_base[1, 0])     # VP/(VP+FN)
recall_smote_re = cm_sm[1, 1] / (cm_sm[1, 1] + cm_sm[1, 0])

tabla_v = pd.DataFrame(
    [["accuracy LDA Iris",     round(acc_lda_re, 4),     val("accuracy_lda_iris"),   "0,9733"],
     ["recall fraude (base)",  round(recall_base_re, 4), val("recall_base_fraude"),  "0,6149"],
     ["recall fraude (SMOTE)", round(recall_smote_re, 4), val("recall_smote_fraude"), "0,8784"]],
    columns=["magnitud", "recomputado", "Excel", "target"])
display(tabla_v)

assert abs(acc_lda_re - val("accuracy_lda_iris")) < 1e-3
assert abs(recall_base_re - val("recall_base_fraude")) < 1e-3
assert abs(recall_smote_re - val("recall_smote_fraude")) < 1e-3
print("assert OK: lo recomputado desde la base coincide con el Excel de contrato (no es un registro suelto).")

**📖 Cómo se lee.** Las tres magnitudes recomputadas desde los datos —**0,9733 / 0,6149 / 0,8784**— coinciden con la hoja `clasificadores_iris`: el Excel **es** producto de la ejecución, no un valor tecleado. Es la contraparte en el cuaderno de el material de referencia de la sesión.

## 10.3 en profundidad — Construcción desde cero (descomposición del alumno) (Sección 7 del cuaderno)

Para probar que se entiende el **flujo del desbalance** —no solo la llamada a la librería— se rearma el pipeline correcto y se contrasta con el error más caro del área:
- **🧱 Flujo CORRECTO:** `split` → **SMOTE SOLO en el train** → ajustar → **métricas en el test real** (que conserva la prevalencia). Reproduce el contrato (recall post-SMOTE **0,8784**).
- **🧱 Flujo con FUGA:** SMOTE sobre **todo** el dataset **antes** del split → métricas **infladas** y engañosas. Es el error #1 de la literatura de fraude (Hayat & Magnier, 2025).

**🔎 Qué hace este código.** Ejecuta **ambos** flujos sobre el fraude: el **correcto** (parte, escala y remuestrea solo el train, evalúa en el test real) y el **con fuga** (escala y remuestrea TODO antes de partir, evalúa en un test remuestreado). **Verifica con `assert`** que el correcto reproduce el contrato y que la fuga **infla** la precisión.

In [ ]:
# 🧱 CONSTRUYE DESDE CERO: flujo correcto del desbalance vs. la FUGA si SMOTE va antes del split. NO toca el Excel.

# --- Flujo CORRECTO: primero se parte, luego se escala y se remuestrea SOLO el train ---
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xf, yf, test_size=0.30, stratify=yf, random_state=42)
esc = StandardScaler().fit(Xh_tr)                                  # escalador ajustado SOLO en train
Xh_tr_s, Xh_te_s = esc.transform(Xh_tr), esc.transform(Xh_te)
Xh_r, yh_r = SMOTE(random_state=42).fit_resample(Xh_tr_s, yh_tr)   # SMOTE SOLO en train
clf_h = LogisticRegression(max_iter=1000).fit(Xh_r, yh_r)
pred_h = (clf_h.predict_proba(Xh_te_s)[:, 1] >= 0.5).astype(int)
recall_honest = float(recall_score(yh_te, pred_h))
prec_honest = float(precision_score(yh_te, pred_h, zero_division=0))
prevalencia_test = float(yh_te.mean())

# --- Flujo con FUGA: escalar y remuestrear TODO el dataset ANTES del split (leakage) ---
esc_full = StandardScaler().fit(Xf)                               # escala sobre TODO -> ya es fuga
Xf_s = esc_full.transform(Xf)
Xf_r, yf_r = SMOTE(random_state=42).fit_resample(Xf_s, yf)        # remuestrea TODO
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xf_r, yf_r, test_size=0.30, stratify=yf_r, random_state=42)
clf_l = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
pred_l = (clf_l.predict_proba(Xl_te)[:, 1] >= 0.5).astype(int)
recall_leak = float(recall_score(yl_te, pred_l))
prec_leak = float(precision_score(yl_te, pred_l, zero_division=0))

print("Flujo CORRECTO (SMOTE solo en train; test con prevalencia real):")
print(f"  prevalencia del test = {prevalencia_test:.5f}   (se conserva el 0,17 % real)")
print(f"  recall = {recall_honest:.4f}   precisión = {prec_honest:.4f}")
print("Flujo con FUGA (SMOTE antes del split; test remuestreado):")
print(f"  recall = {recall_leak:.4f}   precisión = {prec_leak:.4f}   <-- INFLADAS y engañosas")

# Reproduce el contrato del Excel y demuestra la inflación por fuga
meta_b = pd.read_excel(XLSX, sheet_name="clasificadores_iris")
recall_smote_xl = float(meta_b.loc[meta_b["metrica"] == "recall_smote_fraude", "valor"].iloc[0])
assert abs(recall_honest - recall_smote_xl) < 1e-3, "el flujo correcto debe reproducir el recall post-SMOTE del contrato"
assert prec_leak > prec_honest, "la fuga infla la precisión respecto al flujo correcto"
print(f"\nassert OK: el flujo correcto reproduce el contrato (recall {recall_honest:.4f} == Excel {recall_smote_xl}); "
      f"la fuga infla la precisión de {prec_honest:.4f} a {prec_leak:.4f}.")

**📖 Cómo se lee.** ⚠️ El flujo **con fuga** muestra una precisión y un recall **muy altos** (la precisión pasa de **0,0645 a 0,9741**), un espejismo que **colapsa en producción**. Conviene separar los **dos** factores que la fuga mezcla: (1) el **test remuestreado** —se evalúa sobre un test **balanceado** (~50 % de positivos, muchos sintéticos), no sobre el 0,17 % real— y (2) la **síntesis de vecinos** de SMOTE, que contamina el train con puntos interpolados a partir de casos que luego caen en el test. **El número lo domina (1)**: al medir la precisión sobre un test con la mitad de positivos, casi cualquier alerta acierta, así que la precisión aumenta de forma pronunciada por el cambio de prevalencia; la contaminación del train por la síntesis pesa bastante menos. Por eso la regla operativa es **doble**: SMOTE **solo en el train** *y* **evaluar siempre sobre el test con la prevalencia real**. El flujo **correcto** mantiene la precisión real (~0,065) y el recall del contrato (**0,8784**), con un test que conserva el 0,17 % real. La lección: **dónde** se remuestrea —y **sobre qué** se mide— importa más que **si** se remuestrea (la guía de supuestos de la sesión Parte 5.3).

## 10.6 — ¿Cuándo se puede confiar en estos clasificadores? Supuestos: decisiones y condiciones de validez (Sección 8 del cuaderno)

A diferencia del clustering (S07) y las reglas de asociación (S08), esta sesión usa clasificadores **generativos supervisados**, que **sí** asumen una forma para $P(x\mid y)$ (normalidad, igualdad o no de covarianzas, independencia condicional), y toma **decisiones metodológicas** ante el desbalance. La **fuente canónica** es la guía de supuestos de la sesión; aquí se ejecutan **seis diagnósticos** que la ilustran. **Ninguno escribe en el Excel de contrato.**

| Diagnóstico | Qué revela | Fuente |
|---|---|---|
| 8.1 LDA vs QDA (covarianzas) | ¿Las covarianzas por clase difieren lo bastante para justificar QDA? | `SUPUESTOS_S10.md` Parte 2.2 y 3.1 |
| 8.2 Independencia de Naive Bayes | Variables correlacionadas dentro de la clase (viola la independencia) y aun así NB clasifica bien | `SUPUESTOS_S10.md` Parte 4.1 |
| 8.3 Paradoja de la exactitud | Clasificador trivial: accuracy 0,9983, recall 0 | `SUPUESTOS_S10.md` Parte 5.1 |
| 8.4 SMOTE solo en train vs. fuga | El test debe conservar la prevalencia real; la fuga infla | `SUPUESTOS_S10.md` Parte 5.3 |
| 8.5 PR-AUC vs ROC-AUC | La brecha ROC ≫ PR es el síntoma del desbalance | `SUPUESTOS_S10.md` Parte 5.2 |
| 8.6 Umbral y pesos de clase | Mover el umbral / `class_weight` cambia recall/precisión/FP (16 → 1 885) | `SUPUESTOS_S10.md` Parte 5.4 y 5.5 |

**❓ Qué se quiere averiguar.** ¿Varían de veras las tres especies con la misma dispersión y la misma orientación —lo que LDA da por bueno—, y **cuánto cuesta ese supuesto cuando resulta falso**?

- **Qué decide:** si la frontera que se explica al negocio puede ser una recta con coeficientes legibles, o hay que aceptar una curva y renunciar a esa lectura. Es el mismo dilema que aparece al suponer que dos segmentos de clientes tienen la misma variabilidad de gasto.
- **Antes de mirar el resultado:** si el logaritmo del determinante de la covarianza saliera parecido en las tres especies, el supuesto se cumpliría de forma literal y no habría nada que discutir. Si **difiere** pero la ganancia de QDA cabe dentro de la desviación entre pliegues (≈ 0,03), el supuesto es **falso y aun así de bajo costo**: se conserva LDA, por simple y estable. Si QDA ganara varios puntos, el supuesto costaría acierto y habría que pagar los parámetros extra de una covarianza por clase.

**🔎 Qué hace este código (Diagnóstico 8.1 — LDA vs QDA).** Calcula $\log|\Sigma_k|$ y la traza de la covarianza de **cada** especie (si difieren, las covarianzas no son iguales) y compara **LDA vs QDA** bajo el mismo protocolo. Box's M se **nombra** como avanzado. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.1 (LDA vs QDA): comparar covarianzas por clase y la ganancia de QDA — NO escribe en el Excel
iris_s = obtener_iris()
Xi = iris_s[FEATS].values
yi = iris_s["species"].astype("category").cat.codes.values
print("log|Sigma_k| y traza por clase (dispersión/orientación de cada covarianza):")
for c, esp in enumerate(ESPECIES):
    Sc = np.cov(Xi[yi == c].T)
    _signo, logdet = np.linalg.slogdet(Sc)
    print(f"  {esp:11s}: log|Sigma_k| = {logdet:7.3f}   traza = {np.trace(Sc):.3f}")
cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_l = cross_val_score(LinearDiscriminantAnalysis(), Xi, yi, cv=cv_s).mean()
acc_q = cross_val_score(QuadraticDiscriminantAnalysis(), Xi, yi, cv=cv_s).mean()
print(f"\nDiagnóstico empírico (mismo protocolo):  LDA = {acc_l:.4f}   QDA = {acc_q:.4f}   "
      f"ganancia QDA = {acc_q - acc_l:+.4f}")
print("Box's M (test formal de igualdad de covarianzas) se NOMBRA como avanzado: muy sensible al tamaño muestral.")

**📖 Cómo se lee.** Los $\log|\Sigma_k|$ **difieren** entre especies (las covarianzas no son idénticas), pero **QDA apenas mejora** a LDA (0,9800 vs 0,9733, ganancia +0,0067): la heterocedasticidad no es lo bastante marcada para pagar la varianza extra de QDA → **se prefiere LDA** (más simple, frontera interpretable). La decisión **LDA vs QDA es** el diagnóstico de este supuesto (la guía de supuestos de la sesión Parte 2.2 y 3.1).

**❓ Qué se quiere averiguar.** Naive Bayes supone que, dentro de una misma especie, el largo y el ancho del pétalo **no guardan relación**. ¿Es cierto? Y si no lo es, ¿por qué el clasificador acierta de todos modos?

- **Qué decide:** para qué sirve y para qué no un modelo con un supuesto demostradamente falso. La **etiqueta** que produce puede ser correcta y, al mismo tiempo, la **probabilidad** con la que la produce puede no servir para calcular un costo esperado ni para fijar un umbral (por eso el benchmark reporta log-loss).
- **Antes de mirar el resultado:** si la correlación dentro de clase rondara cero, el supuesto se sostendría. Si resulta alta y la accuracy cae de forma marcada, el supuesto era imprescindible. Si resulta alta —por encima de 0,5— y la accuracy se mantiene cerca de la de LDA (0,97 frente a algo más de 0,94), lo que decide la clasificación es el **orden** de las posteriores, no su magnitud: cabe confiar en la etiqueta y desconfiar del número.

**🔎 Qué hace este código (Diagnóstico 8.2 — independencia de Naive Bayes).** Mide la **correlación entre variables dentro de cada clase** (Naive Bayes las supone independientes) y reporta la accuracy de GaussianNB. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.2 (Naive Bayes): correlación entre variables DENTRO de cada clase — NO escribe en el Excel
print("Correlación petal_length vs petal_width DENTRO de cada especie (NB las supone independientes):")
for c, esp in enumerate(ESPECIES):
    r = np.corrcoef(Xi[yi == c, 2], Xi[yi == c, 3])[0, 1]
    print(f"  {esp:11s}: corr = {r:+.3f}")
gnb_acc_diag = cross_val_score(GaussianNB(), Xi, yi, cv=cv_s).mean()
print(f"\nGaussianNB accuracy = {gnb_acc_diag:.4f}  pese a la correlación (viola la independencia y aun así clasifica bien).")
print("Domingos & Pazzani (1997): NB es óptimo bajo pérdida 0-1 aunque se viole la independencia (acierta el ORDEN).")

**📖 Cómo se lee.** Dentro de *versicolor*, largo y ancho de pétalo correlacionan **+0,79**: la independencia condicional **se viola** claramente. Y sin embargo GaussianNB acierta **0,9467**: lo que decide la clasificación es **acertar el orden** de las posteriores, no calibrar su magnitud (Domingos & Pazzani, 1997; la guía de supuestos de la sesión Parte 4.1). ⚠️ Sus `predict_proba`, en cambio, tienden a ser extremas (log-loss alto).

**🔎 Qué hace este código (Diagnóstico 8.3 — paradoja de la exactitud).** Recomputa la accuracy del clasificador **trivial** «todo legítimo» como $1-\text{prevalencia}$ y su recall (0). Verifica con `assert`. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.3 (paradoja de la exactitud): clasificador trivial "todo legítimo" — NO escribe en el Excel
prev = float(yf.mean())
acc_trivial_diag = 1.0 - prev                 # accuracy sin detectar ningún fraude
recall_trivial_diag = 0.0                     # 0 verdaderos positivos
print(f"Prevalencia de fraude = {prev:.5f}  ({prev * 100:.3f} %)")
print(f"Clasificador 'todo legítimo':  accuracy = {acc_trivial_diag:.4f}   recall de fraude = {recall_trivial_diag:.3f}")
print("La precisión del trivial es INDEFINIDA (0/0), no 0.  Un 99,8 % de acierto sin detectar un solo fraude.")
assert abs(acc_trivial_diag - 0.99827) < 1e-3 and recall_trivial_diag == 0.0
print("assert OK: accuracy 0,9983 con recall 0,000 (la exactitud no mide nada útil en 99:1).")

**📖 Cómo se lee.** La accuracy del trivial (**0,9983**) es ≈ $1-\text{prevalencia}$: alta **sin detectar nada**. Por eso **nunca** se elige ni se reporta un modelo por accuracy en desbalance; se fijan de antemano las métricas de la clase rara (la guía de supuestos de la sesión Parte 5.1).

**🔎 Qué hace este código (Diagnóstico 8.4 — SMOTE solo en train vs. fuga).** Recupera del bloque 🧱 los resultados del flujo correcto y del flujo con fuga y verifica que el **test correcto conserva la prevalencia real**. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.4 (SMOTE solo en train vs fuga): el test debe conservar la prevalencia real — NO escribe en el Excel
print("Flujo correcto vs. flujo con fuga (del bloque 🧱 «Construye desde cero»):")
print(f"  CORRECTO  -> recall {recall_honest:.4f} | precisión {prec_honest:.4f} | prevalencia test {prevalencia_test:.5f}")
print(f"  CON FUGA  -> recall {recall_leak:.4f} | precisión {prec_leak:.4f}  (test remuestreado -> métricas infladas)")
assert prevalencia_test < 0.01, "el test del flujo correcto conserva la prevalencia real (<1 %)"
print("\nassert OK: SMOTE SOLO en el train (idealmente en imblearn.Pipeline); el test conserva el 99,83 % : 0,17 %.")

**📖 Cómo se lee.** El flujo correcto evalúa sobre un test con prevalencia **0,00173** (la real); el flujo con fuga evalúa sobre un test **remuestreado** y por eso sus métricas se ven perfectas. Aplicar SMOTE antes del split es una fuga silenciosa (la guía de supuestos de la sesión Parte 5.3).

**🔎 Qué hace este código (Diagnóstico 8.5 — PR-AUC vs ROC-AUC).** Sobre el clasificador base recomputa ROC-AUC y PR-AUC y la línea base (prevalencia), y mide la **brecha**. Verifica con `assert`. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.5 (PR-AUC vs ROC-AUC en desbalance): la brecha es el síntoma — NO escribe en el Excel
roc_v = float(roc_auc_score(yte, proba_base))
pr_v = float(average_precision_score(yte, proba_base))
prev_test = float(yte.mean())
print(f"ROC-AUC = {roc_v:.4f}   (se ve OPTIMISTA: la FPR diluye los FP entre ~85 000 legítimas)")
print(f"PR-AUC  = {pr_v:.4f}   (el número HONESTO en 99:1)")
print(f"Línea base del PR-AUC = prevalencia del test = {prev_test:.4f}   (no 0,5)")
print(f"Brecha ROC - PR = {roc_v - pr_v:.4f}  ->  síntoma del desbalance (Saito & Rehmsmeier, 2015).")
assert roc_v > pr_v, "en 99:1 el ROC-AUC debe superar claramente al PR-AUC"
print("assert OK: ROC-AUC > PR-AUC (por eso se compara y elige por PR-AUC; el ROC es solo contexto).")

**📖 Cómo se lee.** El ROC-AUC (**0,957**) parece excelente, pero el PR-AUC (**0,708**) —cuya línea base es la prevalencia, no 0,5— es el número honesto: la **brecha** entre ambos es el síntoma del desbalance. En 99:1 se decide por **PR-AUC** (la guía de supuestos de la sesión Parte 5.2).

**❓ Qué se quiere averiguar.** Cuando se dice que SMOTE «detecta más fraudes», ¿cambió el modelo o solo cambió **el punto donde se corta**? Y ¿cuánto cuesta cada respuesta en falsas alarmas?

- **Qué decide:** la carga diaria del equipo de revisión. La empresa no elige un modelo, elige un **punto de operación**: cuántas alertas acepta revisar para detectar un fraude más. Ese balance lo fija el costo relativo de un fraude no detectado frente al de una falsa alarma.
- **Antes de mirar el resultado:** con el umbral 0,5 el modelo base deja **16** falsas alarmas y el modelo con SMOTE deja **1 885**. Si al bajar el umbral del modelo base se alcanzara un recall parecido al de SMOTE con muchas menos falsas alarmas, el remuestreo no habría aportado capacidad de discriminación: solo habría desplazado el corte, y el corte se mueve con una línea de código, sin reentrenar. Si hiciera falta un volumen de falsas alarmas comparable, SMOTE sí aporta algo que el umbral no da.

**🔎 Qué hace este código (Diagnóstico 8.6 — umbral y pesos de clase).** Barre el **umbral** del clasificador base y muestra cómo cambian recall, precisión y **FP**; compara los FP en umbral 0,5 (base vs. SMOTE) y nombra `class_weight='balanced'`. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.6 (umbral y pesos de clase): mover el umbral cambia recall/precisión/FP — NO escribe en el Excel
print("Barrido de umbral sobre el clasificador base (mismo modelo, distinto punto de operación):")
print(f"{'umbral':>7s}{'recall':>9s}{'precisión':>11s}{'FP':>8s}")
for u in [0.5, 0.1, 0.01]:
    pu = (proba_base >= u).astype(int)
    fpu = int(confusion_matrix(yte, pu)[0, 1])
    print(f"{u:7.2f}{recall_score(yte, pu):9.4f}{precision_score(yte, pu, zero_division=0):11.4f}{fpu:8d}")
fp_base = int(cm_base[0, 1]); fp_smote = int(cm_sm[0, 1])
print(f"\nPalancas (umbral 0,5):  FP base = {fp_base}  ->  FP con SMOTE = {fp_smote}  (recall sube, precisión cae).")
print("class_weight='balanced' (ver benchmark) pondera la minoritaria sin remuestrear ni añadir fuga.")

**📖 Cómo se lee.** Bajar el umbral **sube el recall** a costa de **más FP** (más falsas alarmas): el punto de operación se fija por el **costo relativo** de FN vs FP (enlace con S09). Comparado con el umbral 0,5, **SMOTE multiplica los FP de 16 a 1 885**; `class_weight='balanced'` es la alternativa sin remuestrear (la guía de supuestos de la sesión Parte 5.4 y 5.5).

## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Se resuelven en parejas; el entregable es individual. Enunciados completos en `evaluacion/drills.docx` (las soluciones son material docente).

1. **La accuracy que induce a error.** Sobre el fraude COMPLETO, construir el clasificador trivial «todo legítimo», reportar su **accuracy** y su **recall de fraude**, y explicar en una frase por qué un 99 % de acierto puede convivir con recall 0. Mostrar la matriz de confusión degenerada.

2. **ROC-AUC vs. PR-AUC.** Con la logística base sobre el fraude, calcular **ROC-AUC** y **PR-AUC (average precision)**, trazar ambas curvas e interpretar por qué difieren tanto en 99:1. Indicar cuál es el «número honesto» y cuál es la **línea base** de cada curva.

3. **Efecto de SMOTE.** Aplicar `SMOTE(random_state=42)` **solo al train**, reentrenar la logística y comparar **recall** y **precision** de fraude antes vs. después (con la matriz de confusión). Discutir el trade-off y en qué situación de negocio conviene. Nombrar el riesgo de aplicar SMOTE antes del split (leakage).

## 10.9 — ¿Qué no se puede afirmar, y qué sigue en S11? Cierre (Sección 10 del cuaderno)

### Entregable evaluable

Construir un **benchmark reproducible de clasificadores** (LDA, QDA, Naive Bayes y una línea base logística) sobre el fraude 99:1, **tratar el desbalance** (SMOTE / `class_weight`), evaluar con las **métricas correctas** (precision, recall, F1, **PR-AUC**, log-loss — no accuracy), elegir el **modelo campeón** con su **umbral** y comunicar la recomendación. Se entrega con la plantilla `plantillas/benchmark_clasificadores.docx` y la guía `plantillas/guia_metricas_desbalance.docx`; enunciado y **rúbrica vigesimal (0–20)** en `evaluacion/entregable.docx`.

### Control corto

Habrá un **control corto** (6–10 preguntas) sobre generativo vs. discriminativo, LDA/QDA/Naive Bayes, la paradoja de la accuracy, ROC-AUC vs. PR-AUC y el efecto de SMOTE. Se alimenta del banco `[S10]`.

### Proyecto integrador

Esta sesión alimenta la fase de **Modelado**: el clasificador de la clase rara (fraude, impago, abandono) con su tratamiento del desbalance y su **umbral por costo** es una pieza directa del proyecto transversal. Cierra el arco de **clasificación** de la Unidad 2.

### Para seguir explorando (las fuentes de actualidad de la sesión)

- **Pérdidas globales por fraude con tarjeta ≈ $33 mil millones** — *GlobeNewswire / The Nilson Report*, 07/01/2026: $33,41 B en 2024; cada punto de recall vale millones (dimensiona el valor de acertar en la clase rara).
- **SMOTE y el trade-off recall/precisión** — *Frontiers in AI* (Albalawi & Dardouri), 08/10/2025: sobre el mismo dataset ULB, la logística con SMOTE llega a **recall 100 % / precisión 24 %** (benchmark publicado, otro pipeline).
- **«Data leakage» que infla los resultados** — *arXiv 2506.02703* (Hayat & Magnier), 03/06/2025: aplicar SMOTE antes del split «logra» 99,9 % de recall engañoso; la rigurosidad pesa más que la complejidad.
- **Revisión sistemática con desbalance original** — *MDPI Computers 14(10):437*, 2025: evaluar preservando el 0,17 % real y con métricas centradas en la minoritaria (PR-AUC).

> **Próximas sesiones (solo se nombran).** S11 series de tiempo, S12 inferencia causal, S13 análisis de supervivencia, S14 sistemas de recomendación. **KNN y SVM** son prerequisito (IA I): aquí solo se referencian.

### Materiales de la sesión que conversan con este cuaderno

- Mapa de celdas ↔ slides: el cuaderno de la sesión.
- Supuestos (fuente canónica): la guía de supuestos de la sesión.
- Validación de la réplica (recomputa desde la base): el material de referencia de la sesión.
- Laboratorio y plantillas: `laboratorio/GUIA_LABORATORIO_S10.docx`, `plantillas/benchmark_clasificadores.docx`, `plantillas/guia_metricas_desbalance.docx`.
- Evaluación y fuentes: `evaluacion/drills.docx`, `evaluacion/entregable.docx`, las fuentes de actualidad de la sesión.